# Supplementary experiments — TCSST-26-766 revision

One notebook that produces new evidence for the 15 reviewer comments that the original run could not fully answer.
It reuses the original data pipeline, masking generator, imputers and seed scheme, so overlapping conditions
reproduce the original `results.csv` numbers (checked automatically in S7 if the original results are attached).

| Block | What it runs | Reviewer comments |
|---|---|---|
| **S1** Importance stability | RF permutation (original 25% holdout) vs repeated 50% holdouts, XGBoost permutation, XGBoost SHAP; fold-to-fold Kendall τ, tier agreement, non-positive share | R1-5, R2-3, R2-5 |
| **S2** Tier design | highest vs original middle member vs average over up to 5 evenly spaced tier members; top / bottom / random / redundancy-matched blocks of equal size; RF basis and XGBoost-SHAP basis | R1-6, R2-4, R2-5, R2-12 |
| **S3** Iterative imputers | max_iter 5 vs 25 (BayesianRidge and RF), all-feature variants that can actually converge, canonical MissForest (Stekhoven & Bühlmann stopping rule), MICE with posterior sampling and m=5 pooled predictions; single-column and 3-column block targets | R1-9, R1-10, R2-6, R2-7 |
| **S4** Classifiers | LR, RF, HGB, XGBoost default × 5 seeds, XGBoost tuned by inner CV on complete training data | R1-11, R2-8, R2-9 |
| **S5** Row cap | 1,500-row cap vs a second independent 1,500-row subsample vs full data for Digits, Spambase, MAGIC, Phoneme; KNN with and without donor cap; 5-NN density | R2-2 |
| **S6** Compute | fixed threads; wall time, CPU time, peak memory per imputer and classifier; hardware record | R1-18 |
| **S7** Reproducibility | archived split indices, masks, probabilities, reconstructed values, fitted models (fold 0); re-run check; match against original results; thread-order test on the exact Spambase case (source of the 0.203-pt gap); imputer seed sensitivity | R1-17, R2-9, R2-10 |
| **S8** Original-run archive | regenerates and archives the split indices and every missingness mask of the **original** core and mismatch experiments (no model fitting), verified against the original mask_checks | R1-17, R2-10 |

**Kaggle setup:** GPU T4 accelerator, *Internet ON* (Ionosphere and Phoneme come from OpenML), attach the same
Banknote, Spambase and MAGIC datasets as the original notebook. Optional: attach your original `results.zip`
as a dataset so S7 can compare against it.

**Runtime:** set `FAST = True` first (≈10–15 min smoke test). The full run is roughly 4–5 h on a T4 session.
S2 evaluates up to 5 members per tier; set `MAX_MEMBERS['permutation'] = None` to use every member (≈+1.5 h).
Every (experiment, dataset, fold) group is checkpointed; if the session stops, re-run all cells and it resumes.

In [1]:
# ===================== CONFIGURATION =====================
import os
FAST = False                 # True = quick smoke test (2 datasets, 2 folds, fewer settings)
N_THREADS = 4                # fixed CPU thread budget for every library (R1-18)
for _v in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[_v] = str(N_THREADS)

RUN = {'S1': True, 'S2': True, 'S3': True, 'S4': True, 'S5': True, 'S6': True, 'S7': True, 'S8': True}
ALLOW_OPENML = True          # Ionosphere (59) and Phoneme (1489)
FILE_OVERRIDES = {}          # e.g. {'spambase': '/kaggle/input/.../spambase.csv'}
ORIGINAL_RESULTS = None      # optional path to original study_results/<run-id> folder; auto-searched if None

SEED = 2026; FOLDS = 3; REPEATS = 2; MASK_SEED = 17; RATE = 0.2; MAX_ROWS = 1500
FOLD_IDS = [0, 1] if FAST else list(range(FOLDS * REPEATS))
MAX_DATASETS = 2 if FAST else 7
IMP_REPEATS = 1 if FAST else 3          # repeated 2-fold inner splits for stable importance (=6 holdouts)
IMP_PERM_REPEATS = 2 if FAST else 3
# tier members evaluated per tier: up to 5, evenly spaced and always including the highest and original middle member
MAX_MEMBERS = {'permutation': 3, 'xgb_shap': 3} if FAST else {'permutation': 5, 'xgb_shap': 5}  # None = every member
XGB_SEEDS = 2 if FAST else 5
TUNE_CONFIGS = 3 if FAST else 12
MICE_M = 3 if FAST else 5
SEED_SENSITIVITY_SEEDS = 3 if FAST else 10
THREAD_REPEATS = 3 if FAST else 8      # repeated identical runs for the thread-order test (R2-10)
DETERMINISTIC_PREDICT = True           # forests predict single-threaded so new artifacts are bit-reproducible
REPRO_SAMPLE = 6 if FAST else 40
SAVE_MODEL_FOLDS = [0]                  # fitted imputers/classifiers archived for these folds ...
SAVE_MODEL_EXPS = ['S4', 'S5']          # ... and these experiments (others are rebuilt exactly from splits+masks+seeds)
BOOTSTRAPS = 2000

# Fingerprints of the capped datasets in the original run (config.json); checked below.
ORIGINAL_FINGERPRINTS = {
    'breast_cancer': '4865f8b0c2cb73ba79f9309248402c6be8c450c78446061c6bf77e5d5d7ee32e',
    'digits': '56385459eb4e4131b8df0e677eecb5ca52deec0971ac6f7801c3d021f74565e3',
    'banknote': '806dab2653d5a5ebd0cc8428ed2fc146b641d9ac904eb5eeaceb63ceee8dbb12',
    'spambase': '068911ee552e466016332a04433b77f9309a9844cefcbd17b5b4cdfd977d33dd',
    'magic': '4d37a5372d7225bb8ea1fc331c1d64ee02296e453a1c0d3a1ab47b47d2080968',
    'ionosphere': '2b5f58dd91622179a450961b3119398b26afc20f3b410fc702c4216314270206',
    'phoneme': '668c2159814f3e3e274b12ebf99e37eb73dccd50017ec480fc76e79c5bc3e3d6'}

In [2]:
import sys, json, time, hashlib, warnings, platform, subprocess, tracemalloc, itertools, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, scipy, sklearn, joblib, psutil
from scipy.special import expit
from scipy.stats import kendalltau, spearmanr, wilcoxon
from sklearn.datasets import load_breast_cancer, load_digits, fetch_openml
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from threadpoolctl import threadpool_limits
try:
    import xgboost as xgb
except ImportError:
    xgb = None
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice')
warnings.filterwarnings('ignore', message='An input array is constant')
warnings.filterwarnings('ignore', message='.*Falling back to prediction using DMatrix.*')
_limits = threadpool_limits(limits=N_THREADS)

# FAST and full runs write to separate folders so a full run never resumes from reduced FAST checkpoints
OUT = (Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd() / 'output') / ('supplementary_FAST' if FAST else 'supplementary')
OUT.mkdir(parents=True, exist_ok=True)
ART = OUT / 'artifacts'; ART.mkdir(exist_ok=True)
SUM = OUT / 'summary'; SUM.mkdir(exist_ok=True)

try:
    _g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                        capture_output=True, text=True, timeout=10)
    GPU_INFO = _g.stdout.strip() if _g.returncode == 0 else ''
except (OSError, subprocess.TimeoutExpired):
    GPU_INFO = ''
GPU = bool(GPU_INFO) and xgb is not None
if xgb is None:
    print('WARNING: xgboost not installed - "xgb" falls back to HistGradientBoosting (smoke test only).')
VERSIONS = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
            'xgboost': getattr(xgb, '__version__', 'missing')}
print(json.dumps(VERSIONS, indent=1)); print('GPU:', GPU_INFO or 'none')


def json_scalar(v):
    return v.item() if isinstance(v, np.generic) else str(v)


def stable_seed(*parts):
    """Identical to the original notebook: SHA-256 of the JSON-encoded parts."""
    return int(hashlib.sha256(json.dumps(parts, default=json_scalar).encode()).hexdigest()[:8], 16)


def rid(*parts):
    return hashlib.sha256(json.dumps(parts, default=json_scalar).encode()).hexdigest()[:20]


def atomic_csv(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp'); df.to_csv(tmp, index=False); os.replace(tmp, path)


def atomic_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj, indent=1, default=json_scalar)); os.replace(tmp, path)

{
 "python": "3.13.16",
 "numpy": "2.1.3",
 "pandas": "2.3.3",
 "scipy": "1.16.3",
 "sklearn": "1.6.1",
 "xgboost": "3.4.1"
}
GPU: Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


## Data (same loaders, cleaning and stratified 1,500-row cap as the original; full versions kept for S5)

In [3]:
INPUT_ROOT = Path('/kaggle/input')
ATTACHED = sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file()) if INPUT_ROOT.exists() else []


def find_input(name, tokens, suffixes=('.csv', '.data', '.txt')):
    if name in FILE_OVERRIDES: return Path(FILE_OVERRIDES[name])
    c = [p for p in ATTACHED if p.suffix.lower() in suffixes and any(t.lower() in str(p).lower() for t in tokens)
         and not any(t in p.name.lower() for t in ['names', 'metadata', 'readme', 'test'])
         and 'study_results' not in str(p)]
    if len(c) > 1:
        pref = [p for p in c if p.suffix.lower() == '.csv']
        if len(pref) == 1: return pref[0]
        raise ValueError(f'Ambiguous {name}: {c}; set FILE_OVERRIDES')
    return c[0] if c else None


def clean_frame(df):
    df = df.copy()
    df.columns = [str(c).strip().lower().replace('.', '_').replace(' ', '_') for c in df.columns]
    for c in df.select_dtypes(include=['object', 'string', 'category']):
        df[c] = df[c].astype(object).map(lambda v: v.strip() if isinstance(v, str) else v)
        df[c] = df[c].replace({'?': np.nan, '': np.nan})
        num = pd.to_numeric(df[c], errors='coerce')
        if num.notna().sum() == df[c].notna().sum(): df[c] = num
    return df.replace([np.inf, -np.inf], np.nan)


def _is_number(x):
    try: float(str(x)); return True
    except ValueError: return False


def drop_leading_index(df, n):
    if df.shape[1] == n + 1:
        first = pd.to_numeric(df.iloc[:, 0], errors='coerce'); label = str(df.columns[0]).strip().lower()
        seq = bool(first.notna().all() and len(first) > 1 and (first.diff().dropna() == 1).all())
        if label.startswith('unnamed') or label in ('', 'index', 'id') or seq:
            return df.iloc[:, 1:].reset_index(drop=True)
    return df


def read_registered(name, path):
    df = pd.read_csv(path, na_values=['?', ' ?'], skipinitialspace=True)
    expected = {'banknote': 5, 'spambase': 58, 'magic': 11}
    if path.suffix.lower() == '.data' or all(_is_number(c) for c in df.columns[:-1]):
        df = pd.read_csv(path, header=None, na_values=['?'], skipinitialspace=True)
        df.columns = [f'x{i}' for i in range(df.shape[1] - 1)] + ['target']
    df = drop_leading_index(df, expected[name])
    if df.shape[1] != expected[name]: raise ValueError(f'{name}: expected {expected[name]} columns')
    df = clean_frame(df)
    target = next((c for c in ['class', 'spam', 'target'] if c in df.columns), df.columns[-1])
    y = df.pop(target)
    return df, y


DATA = {}; AUDIT = []


def register(name, X, y, source):
    """Original Group-A rules; keeps capped ('cap') and uncapped ('full') versions."""
    X = clean_frame(pd.DataFrame(X)).reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    ok = y.notna(); X = X.loc[ok].reset_index(drop=True); y = y.loc[ok].reset_index(drop=True)
    j = X.copy(); j['__label__'] = y; keep = ~j.duplicated()
    X = X.loc[keep].reset_index(drop=True); y = y.loc[keep].reset_index(drop=True)
    n0 = len(X)
    if X.isna().any().any() or not all(pd.api.types.is_numeric_dtype(X[c]) for c in X):
        AUDIT.append({'dataset': name, 'status': 'excluded', 'reason': 'missing or non-numeric'}); return
    versions = {}
    full_X, full_y = X, y
    if len(X) > MAX_ROWS:
        ch, _ = train_test_split(np.arange(len(X)), train_size=MAX_ROWS, stratify=y, random_state=SEED)
        ch = np.sort(ch); X = X.iloc[ch].reset_index(drop=True); y = y.iloc[ch].reset_index(drop=True)
    alt = None
    if n0 > MAX_ROWS:   # second, independent stratified subsample (R2-2 sampling-bias check)
        ch2, _ = train_test_split(np.arange(n0), train_size=MAX_ROWS, stratify=full_y, random_state=SEED + 1)
        ch2 = np.sort(ch2); alt = (full_X.iloc[ch2].reset_index(drop=True), full_y.iloc[ch2].reset_index(drop=True))
    for tag, XX, yy0 in [('cap', X, y), ('cap_alt', *(alt or (None, None))), ('full', full_X, full_y)]:
        if tag != 'cap' and n0 <= MAX_ROWS: continue
        enc = LabelEncoder(); yy = enc.fit_transform(yy0.astype(str))
        fp = hashlib.sha256(pd.util.hash_pandas_object(XX, index=True).values.tobytes() + yy.tobytes()).hexdigest()
        versions[tag] = {'X': XX.to_numpy(float), 'y': yy, 'columns': list(XX.columns), 'fingerprint': fp}
    DATA[name] = {'versions': versions, 'original_rows': n0, 'source': source}
    fp = versions['cap']['fingerprint']
    AUDIT.append({'dataset': name, 'status': 'included', 'original_rows': n0, 'capped_rows': len(X),
                  'source': source, 'fingerprint': fp,
                  'matches_original_run': fp == ORIGINAL_FINGERPRINTS.get(name)})


for nm, loader in [('breast_cancer', load_breast_cancer), ('digits', load_digits)]:
    d = loader(as_frame=True); register(nm, d.data, d.target, f'sklearn.{loader.__name__}')
for nm, tokens in [('banknote', ['banknote', 'bank-note-authentication']), ('spambase', ['spambase']),
                   ('magic', ['magic-gamma', 'magic04', 'magic_gamma'])]:
    try:
        p = find_input(nm, tokens)
        if p is None: AUDIT.append({'dataset': nm, 'status': 'unavailable', 'reason': 'attach Kaggle dataset'}); continue
        X, y = read_registered(nm, p); register(nm, X, y, str(p))
    except Exception as e:
        AUDIT.append({'dataset': nm, 'status': 'failed', 'reason': str(e)})
if ALLOW_OPENML:
    for nm, did in [('ionosphere', 59), ('phoneme', 1489)]:
        try:
            d = fetch_openml(data_id=did, as_frame=True, n_retries=1, delay=.1)
            register(nm, d.data, d.target, f'https://www.openml.org/d/{did}')
        except Exception as e:
            AUDIT.append({'dataset': nm, 'status': 'failed', 'reason': str(e)[:120]})
ORDER = ['breast_cancer', 'banknote', 'digits', 'spambase', 'magic', 'ionosphere', 'phoneme']
DATASETS = [n for n in ORDER if n in DATA][:MAX_DATASETS]
AUDIT_DF = pd.DataFrame(AUDIT); atomic_csv(AUDIT_DF, SUM / 'dataset_audit.csv'); display(AUDIT_DF)
assert DATASETS, 'No datasets available'
print('Datasets:', DATASETS)


def dkey(name, cap):
    """Seed key: identical to the original for capped data."""
    return name if cap == 'cap' else f'{name}@full'


_SPLITS = {}


def splits(name, cap):
    if (name, cap) not in _SPLITS:
        v = DATA[name]['versions'][cap]
        cv = RepeatedStratifiedKFold(n_splits=FOLDS, n_repeats=REPEATS, random_state=SEED)
        sp = list(cv.split(v['X'], v['y'])); _SPLITS[(name, cap)] = sp
        path = ART / 'splits' / f'{name}_{cap}.npz'
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(path, **{f'train_{i}': a for i, (a, b) in enumerate(sp)},
                                **{f'test_{i}': b for i, (a, b) in enumerate(sp)})
    return _SPLITS[(name, cap)]


def fold_data(name, cap, fold):
    v = DATA[name]['versions'][cap]; tr, te = splits(name, cap)[fold]
    return v['X'][tr], v['y'][tr], v['X'][te], v['y'][te]

,dataset,status,original_rows,capped_rows,source,fingerprint,matches_original_run
0,breast_cancer,included,569,569,sklearn.load_breast_cancer,4865f8b0c2cb73ba79f9309248402c6be8c450c7844606...,True
1,digits,included,1797,1500,sklearn.load_digits,56385459eb4e4131b8df0e677eecb5ca52deec0971ac6f...,True
2,banknote,included,1348,1348,/kaggle/input/datasets/ritesaluja/bank-note-au...,806dab2653d5a5ebd0cc8428ed2fc146b641d9ac904eb5...,True
3,spambase,included,4210,1500,/kaggle/input/datasets/kaggleprollc/spam-email...,5b237d8a2fc17eeaa1a8e1f97a40a3b336f988d2162cc5...,False
4,magic,included,18905,1500,/kaggle/input/datasets/abhinavmangalore/magic-...,4d37a5372d7225bb8ea1fc331c1d64ee02296e453a1c0d...,True
5,ionosphere,included,350,350,https://www.openml.org/d/59,2b5f58dd91622179a450961b3119398b26afc20f3b410f...,True
6,phoneme,included,5395,1500,https://www.openml.org/d/1489,668c2159814f3e3e274b12ebf99e37eb73dccd50017ec4...,True


Datasets: ['breast_cancer', 'banknote', 'digits', 'spambase', 'magic', 'ionosphere', 'phoneme']


## Shared machinery: masking (original), imputers (original + new variants), classifiers, metrics

In [4]:
STRENGTHS = {'mnar_mild': .75, 'mnar_moderate': 2., 'mnar_strong': 6.}


class _SerialPredict:
    """Forest whose prediction averages trees in a fixed order. scikit-learn sums tree outputs from
    several threads, so the summation order (and the last bits) can vary between runs - this is what
    made the original Spambase strong-MNAR RF-iterative record differ by 0.2 points."""
    def predict(self, X):
        if not DETERMINISTIC_PREDICT: return super().predict(X)
        nj = self.n_jobs; self.n_jobs = 1
        try: return super().predict(X)
        finally: self.n_jobs = nj

    def predict_proba(self, X):
        if not DETERMINISTIC_PREDICT: return super().predict_proba(X)
        nj = self.n_jobs; self.n_jobs = 1
        try: return super().predict_proba(X)
        finally: self.n_jobs = nj


class DetRFRegressor(_SerialPredict, RandomForestRegressor):
    pass


class DetRFClassifier(_SerialPredict, RandomForestClassifier):
    def predict_proba(self, X):
        return _SerialPredict.predict_proba(self, X)


def calibrate_intercept(scores, rate):
    lo, hi = -80., 80.
    for _ in range(80):
        mid = (lo + hi) / 2
        if expit(np.clip(scores + mid, -80, 80)).mean() < rate: lo = mid
        else: hi = mid
    return (lo + hi) / 2


class MaskGenerator:
    """Unchanged from the original notebook (rows masked jointly across `cols`; MNAR driven by cols[0])."""
    def __init__(self, mechanism, rate, cols): self.mechanism = mechanism; self.rate = float(rate); self.cols = list(cols)

    def fit(self, A, y, C):
        if self.rate == 0: self.target = self.cols[0]; self.driver = None; self.driver_corr = np.nan; return self
        self.target = self.cols[0]; self.driver = None; self.driver_corr = np.nan; m = self.mechanism
        if m.startswith('mar_'):
            cand = [j for j in range(A.shape[1]) if j not in self.cols and np.std(A[:, j]) > 1e-10]
            pick = max if m == 'mar_redundant' else min
            self.driver = pick(cand, key=lambda j: C[self.target, j]); self.driver_corr = float(C[self.target, self.driver])
            idx = self.driver; self.mu = float(A[:, idx].mean()); self.sd = max(float(A[:, idx].std()), 1e-8)
            self.slope = 2.; self.intercept = calibrate_intercept(self.slope * (A[:, idx] - self.mu) / self.sd, self.rate)
        elif m in STRENGTHS:
            idx = self.target; self.mu = float(A[:, idx].mean()); self.sd = max(float(A[:, idx].std()), 1e-8)
            self.slope = STRENGTHS[m]; self.intercept = calibrate_intercept(self.slope * (A[:, idx] - self.mu) / self.sd, self.rate)
        elif m == 'label':
            counts = np.bincount(y); mino = int(counts.argmin())
            self.label_scores = {int(c): (2.5 if c == mino else -.5) for c in np.unique(y)}
            self.intercept = calibrate_intercept(np.array([self.label_scores[int(c)] for c in y]), self.rate)
        elif m == 'mnar_censor':
            v = A[:, self.target]; self.threshold = float(np.quantile(v, 1 - self.rate))
            above = np.mean(v > self.threshold); equal = np.mean(v == self.threshold)
            self.tie_probability = float(np.clip((self.rate - above) / max(equal, 1e-12), 0, 1))
        elif m not in ('mcar', 'block_top', 'block_random'):
            raise ValueError(m)
        return self

    def probabilities(self, A, y):
        m = self.mechanism
        if m.startswith('mar_'): return expit(self.intercept + self.slope * (A[:, self.driver] - self.mu) / self.sd)
        if m in STRENGTHS: return expit(self.intercept + self.slope * (A[:, self.target] - self.mu) / self.sd)
        if m == 'label': return expit(self.intercept + np.array([self.label_scores.get(int(c), 0) for c in y]))
        if m == 'mnar_censor':
            v = A[:, self.target]
            return np.where(v > self.threshold, 1., np.where(v == self.threshold, self.tie_probability, 0.))
        return np.full(len(A), self.rate)

    def sample(self, A, y, seed):
        if self.rate == 0: return np.zeros(A.shape, dtype=bool), np.zeros(len(A))
        rng = np.random.default_rng(seed); probs = self.probabilities(A, y)
        rows = rng.random(len(A)) < probs
        mask = np.zeros(A.shape, dtype=bool); mask[:, self.cols] = rows[:, None]
        return mask, probs



class MissForest:
    """Canonical MissForest (Stekhoven & Buehlmann 2012): mean start, columns by increasing missingness,
    RF(100 trees, mtry=sqrt(p)) per column, stop when the normalised change increases for the first time
    and return the previous iterate; max 10 iterations. Inductive use: the accepted iterations' forests
    are replayed on test rows."""
    def __init__(self, seed, max_iter=10, n_estimators=100):
        self.seed = seed; self.max_iter = max_iter; self.n_estimators = n_estimators

    def fit_transform(self, Z):
        miss = np.isnan(Z); p = Z.shape[1]
        self.init = np.nan_to_num(np.nanmean(Z, axis=0))
        self.cols = [int(j) for j in np.argsort(miss.sum(0), kind='stable') if miss[:, j].any()]
        X = np.where(miss, self.init, Z); self.history = []; prev = np.inf; self.stopped = 'max_iter'
        for it in range(self.max_iter):
            old = X.copy(); models = {}
            for j in self.cols:
                o = ~miss[:, j]; others = [k for k in range(p) if k != j]
                rf = DetRFRegressor(n_estimators=self.n_estimators, max_features='sqrt',
                                           random_state=stable_seed(self.seed, it, j), n_jobs=N_THREADS)
                rf.fit(X[o][:, others], Z[o, j]); X[miss[:, j], j] = rf.predict(X[miss[:, j]][:, others]); models[j] = rf
            delta = np.sum((X - old) ** 2) / max(np.sum(X ** 2), 1e-12)
            if delta > prev:
                X = old; self.stopped = 'delta_increase'; break
            self.history.append(models); prev = delta
        self.n_iter_ = len(self.history)
        return X

    def transform(self, Z):
        miss = np.isnan(Z); p = Z.shape[1]; X = np.where(miss, self.init, Z)
        for models in self.history:
            for j in self.cols:
                r = miss[:, j]
                if r.any():
                    others = [k for k in range(p) if k != j]; X[r, j] = models[j].predict(X[r][:, others])
        return X


IMPUTER_INFO = {'mean': 'original', 'median': 'original', 'median_indicator': 'original', 'knn': 'original (1,000 donor cap)',
                'knn_full': 'KNN without donor cap', 'iterative': 'original BayesianRidge, max_iter=5',
                'iterative_mi25': 'BayesianRidge, max_iter=25', 'rf_iterative': 'original RF-IterativeImputer, max_iter=5',
                'rf_iterative_mi25': 'RF-IterativeImputer, max_iter=25', 'missforest': 'canonical MissForest',
                'iterative_allfeat': 'BayesianRidge, all features as predictors, max_iter=25 (can converge)',
                'rf_iterative_allfeat': 'RF-IterativeImputer, all features as predictors, max_iter=25 (can converge)',
                'mice5': 'MICE, sample_posterior=True, max_iter=10, m pooled', 'native': 'XGBoost/HGB native NaN'}


def impute(method, Atr, Bte, seed):
    """Returns (list of (train, test, ind_train, ind_test) completions, info, fitted object)."""
    mu = np.nan_to_num(np.nanmean(Atr, axis=0)); sd = np.nanstd(Atr, axis=0)
    sd = np.where(np.isfinite(sd) & (sd > 1e-8), sd, 1.)
    Za, Zb = (Atr - mu) / sd, (Bte - mu) / sd; info = {'n_iter': np.nan, 'max_iter': np.nan, 'converged': np.nan,
                                                      'warning': ''}
    ind = (np.isnan(Atr).astype(float), np.isnan(Bte).astype(float)) if method == 'median_indicator' else (None, None)
    if method == 'native':
        return [(Atr.copy(), Bte.copy(), None, None)], info, None
    p = Za.shape[1]; skip = bool(np.isnan(Za).any()); nnf = min(12, p - 1) if p > 1 else None
    outs = []; obj = None
    with warnings.catch_warnings(record=True) as cap:
        warnings.simplefilter('always')
        if method in ('mean', 'median', 'median_indicator'):
            obj = SimpleImputer(strategy='median' if method != 'mean' else 'mean', keep_empty_features=True).fit(Za)
            outs = [(obj.transform(Za), obj.transform(Zb))]
        elif method in ('knn', 'knn_full'):
            donors = Za
            if method == 'knn' and len(Za) > 1000:
                donors = Za[np.random.default_rng(seed).choice(len(Za), 1000, replace=False)]
            obj = KNNImputer(n_neighbors=5, weights='distance', keep_empty_features=True).fit(donors)
            outs = [(obj.transform(Za), obj.transform(Zb))]
        elif method in ('iterative', 'iterative_mi25', 'rf_iterative', 'rf_iterative_mi25', 'iterative_allfeat',
                        'rf_iterative_allfeat'):
            RF = DetRFRegressor if DETERMINISTIC_PREDICT else RandomForestRegressor
            est = BayesianRidge() if method.startswith('iterative') else RF(
                n_estimators=40, max_depth=8, min_samples_leaf=3, max_features=.8, n_jobs=N_THREADS, random_state=seed)
            mi = 5 if method in ('iterative', 'rf_iterative') else 25
            obj = IterativeImputer(estimator=est, max_iter=mi, tol=.01, initial_strategy='median', skip_complete=skip,
                                   n_nearest_features=None if method.endswith('allfeat') else nnf,
                                   keep_empty_features=True, random_state=seed).fit(Za)
            outs = [(obj.transform(Za), obj.transform(Zb))]
            info.update(n_iter=obj.n_iter_, max_iter=mi, converged=bool(obj.n_iter_ < mi))
        elif method == 'missforest':
            obj = MissForest(seed); ta = obj.fit_transform(Za); outs = [(ta, obj.transform(Zb))]
            info.update(n_iter=obj.n_iter_, max_iter=obj.max_iter, converged=obj.stopped == 'delta_increase',
                        stop_rule=obj.stopped)
        elif method == 'mice5':
            obj = []
            for m in range(MICE_M):
                ii = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, max_iter=10, initial_strategy='median',
                                      skip_complete=skip, keep_empty_features=True,
                                      random_state=stable_seed(seed, 'mice', m)).fit(Za)
                outs.append((ii.transform(Za), ii.transform(Zb))); obj.append(ii)
            info.update(n_iter=10, max_iter=10, converged=np.nan)
        else:
            raise ValueError(method)
    info['warning'] = ' | '.join(sorted(set(str(w.message)[:80] for w in cap if 'Early stopping' in str(w.message)
                                            or 'Convergence' in type(w.message).__name__)))
    res = [(a * sd + mu, b * sd + mu, ind[0], ind[1]) for a, b in outs]
    return res, info, obj


SPEC_NATIVE_OK = lambda spec: spec.startswith('xgb') or spec == 'hgb'
XGB_DEFAULT = dict(n_estimators=120, max_depth=4, learning_rate=.1, subsample=.9, colsample_bytree=.9, reg_lambda=1.)


def make_clf(spec, seed, n_classes, params=None, device=None):
    base = spec.split('_s')[0] if spec.startswith('xgb_s') else spec
    if base == 'lr': return make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=1000, random_state=seed))
    if base == 'rf': return (DetRFClassifier if DETERMINISTIC_PREDICT else RandomForestClassifier)(
        n_estimators=120, min_samples_leaf=2, n_jobs=N_THREADS, random_state=seed)
    if base == 'hgb' or (base.startswith('xgb') and xgb is None):
        return HistGradientBoostingClassifier(max_iter=80, max_leaf_nodes=15, learning_rate=.1, l2_regularization=1.,
                                              early_stopping=False, random_state=seed % (2 ** 31))
    if base.startswith('xgb'):
        kw = dict(XGB_DEFAULT if params is None else params)
        return xgb.XGBClassifier(**kw, tree_method='hist', device=device or ('cuda' if GPU else 'cpu'),
                                 n_jobs=N_THREADS, random_state=seed, verbosity=0,
                                 objective='binary:logistic' if n_classes == 2 else 'multi:softprob',
                                 eval_metric='logloss' if n_classes == 2 else 'mlogloss')
    raise ValueError(spec)


def spec_seed(spec, key, fold):
    """Original model seed for lr/rf/hgb/xgb; new seeds for extra XGBoost seeds and the tuned model."""
    if spec.startswith('xgb_s'): return stable_seed(SEED, key, int(fold), 'xgb', 'extra_seed', int(spec[5:]))
    return stable_seed(SEED, key, int(fold), spec)


def calculate_metrics(y, P, minority):
    P = np.clip(P, 1e-12, 1); P = P / P.sum(axis=1, keepdims=True); pred = P.argmax(axis=1); k = P.shape[1]
    r = {'accuracy': accuracy_score(y, pred), 'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
         'balanced_accuracy': balanced_accuracy_score(y, pred),
         'minority_f1': f1_score(y, pred, labels=[minority], average='macro', zero_division=0)}
    try: r['roc_auc'] = roc_auc_score(y, P[:, 1]) if k == 2 else roc_auc_score(y, P, multi_class='ovr', average='macro')
    except ValueError: r['roc_auc'] = np.nan
    return r


METRICS = ['accuracy', 'macro_f1', 'balanced_accuracy', 'minority_f1', 'roc_auc']
_TUNED = {}


def tuned_params(name, cap, fold, A, y):
    """Random search (TUNE_CONFIGS) with inner 3-fold CV on the complete training fold only."""
    key = f'{name}_{cap}_{fold}'; path = OUT / 'tuning' / f'{key}.json'
    if key in _TUNED: return _TUNED[key]
    if path.exists(): _TUNED[key] = json.loads(path.read_text()); return _TUNED[key]
    rng = np.random.default_rng(stable_seed(SEED, dkey(name, cap), fold, 'tuning'))
    space = {'n_estimators': [100, 200, 400], 'max_depth': [3, 4, 6, 8], 'learning_rate': [.03, .1, .3],
             'subsample': [.7, .9, 1.], 'colsample_bytree': [.6, .8, 1.], 'min_child_weight': [1, 3, 5],
             'reg_lambda': [.5, 1., 5.]}
    configs = [dict(XGB_DEFAULT)] + [{k: v[rng.integers(len(v))] for k, v in space.items()} for _ in range(TUNE_CONFIGS - 1)]
    inner = StratifiedKFold(3, shuffle=True, random_state=stable_seed(SEED, dkey(name, cap), fold, 'tuning_cv'))
    rows = []
    for ci, cfg in enumerate(configs):
        s = []
        for a, b in inner.split(A, y):
            clf = make_clf('xgb_tuned', 0, len(np.unique(y)), cfg); clf.fit(A[a], y[a])
            s.append(f1_score(y[b], clf.predict_proba(A[b]).argmax(1), average='macro'))
        rows.append({'config': ci, 'cv_macro_f1': float(np.mean(s)), **cfg})
    best = max(rows, key=lambda r: r['cv_macro_f1'])
    res = {k: (v.item() if isinstance(v, np.generic) else v) for k, v in best.items() if k in space}
    res['_cv_macro_f1'] = best['cv_macro_f1']; res['_default_cv_macro_f1'] = rows[0]['cv_macro_f1']
    atomic_json(res, path); _TUNED[key] = res
    return res


def tuned_cfg(name, cap, fold):
    key = f'{name}_{cap}_{fold}'
    if key not in _TUNED: _TUNED[key] = json.loads((OUT / 'tuning' / f'{key}.json').read_text())
    return {k: v for k, v in _TUNED[key].items() if not k.startswith('_')}


def fit_predict(spec, Xa, y, Xb, seed, name, cap, fold):
    params = tuned_cfg(name, cap, fold) if (spec == 'xgb_tuned' and xgb is not None) else None
    clf = make_clf(spec, seed, len(np.unique(y)), params)
    clf.fit(Xa, y); return clf.predict_proba(Xb), clf


_BASE = {}


def baseline(name, cap, fold, spec, A, y, B, z):
    key = (name, cap, fold, spec)
    if key not in _BASE:
        path = OUT / 'baselines' / f'{name}_{cap}_{fold}_{spec}.json'
        if path.exists(): _BASE[key] = json.loads(path.read_text())
        else:
            P, _ = fit_predict(spec, A.astype(np.float32), y, B.astype(np.float32), spec_seed(spec, dkey(name, cap), fold),
                               name, cap, fold)
            _BASE[key] = calculate_metrics(z, P, int(np.bincount(y).argmin())); atomic_json(_BASE[key], path)
    return _BASE[key]

## Feature importance with several bases (S1 computation; reused by S2–S5)

In [5]:
BASES = ['permutation', 'rf_perm_rep', 'xgb_perm_rep', 'xgb_shap']


def xgb_shap(clf, X):
    if xgb is None or not hasattr(clf, 'get_booster'):
        return np.full(X.shape[1], np.nan)
    c = clf.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
    c = np.abs(c)
    if c.ndim == 3: c = c.sum(axis=1)
    return c[:, :-1].mean(axis=0)


def importance(name, cap, fold, full=True):
    """'permutation' replicates the original exactly (RF 100 trees, 25% holdout, 5 repeats, same seed).
    The repeated bases average IMP_REPEATS x 2 holdouts of 50%."""
    path = OUT / 'importance' / f'{name}_{cap}_{fold}.csv'
    if path.exists():
        fr = pd.read_csv(path)
        if full is False or 'xgb_shap' in fr: return fr
    A, y, _, _ = fold_data(name, cap, fold); key = dkey(name, cap); p = A.shape[1]
    seed = stable_seed(SEED, key, fold, 'importance')
    a, b = train_test_split(np.arange(len(y)), test_size=.25, stratify=y, random_state=seed)
    rf = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=2, n_jobs=N_THREADS, random_state=seed)
    rf.fit(A[a], y[a])
    pi = permutation_importance(rf, A[b], y[b], scoring='f1_macro', n_repeats=5, random_state=seed, n_jobs=1)
    C = pd.DataFrame(A).corr().abs().fillna(0).to_numpy(copy=True); np.fill_diagonal(C, 0)
    fr = pd.DataFrame({'feature_id': np.arange(p), 'feature': DATA[name]['versions'][cap]['columns'],
                       'permutation': pi.importances_mean, 'max_abs_corr': C.max(axis=1),
                       'eligible': np.std(A, axis=0) > 1e-10})
    if full:
        rep = {k: [] for k in BASES[1:]}
        cv = RepeatedStratifiedKFold(n_splits=2, n_repeats=IMP_REPEATS, random_state=stable_seed(SEED, key, fold, 'imp_rep'))
        for h, (ia, ib) in enumerate(cv.split(A, y)):
            s = stable_seed(SEED, key, fold, 'imp_rep', h)
            r = RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=N_THREADS, random_state=s).fit(A[ia], y[ia])
            rep['rf_perm_rep'].append(permutation_importance(r, A[ib], y[ib], scoring='f1_macro', n_repeats=IMP_PERM_REPEATS,
                                                             random_state=s, n_jobs=1).importances_mean)
            x = make_clf('xgb', s, len(np.unique(y))).fit(A[ia].astype(np.float32), y[ia])
            rep['xgb_perm_rep'].append(permutation_importance(x, A[ib].astype(np.float32), y[ib], scoring='f1_macro',
                                                              n_repeats=IMP_PERM_REPEATS, random_state=s, n_jobs=1).importances_mean)
            sh = xgb_shap(x, A[ib].astype(np.float32))
            rep['xgb_shap'].append(sh if np.isfinite(sh).all() else rep['xgb_perm_rep'][-1])
        for k, v in rep.items():
            V = np.array(v); fr[k] = V.mean(0); fr[f'{k}_sd'] = V.std(0)
            taus = [kendalltau(V[i], V[j])[0] for i, j in itertools.combinations(range(len(V)), 2)]
            fr[f'{k}_inner_tau'] = np.nanmean(taus) if taus else np.nan
    fr['dataset'] = name; fr['cap'] = cap; fr['fold'] = fold
    atomic_csv(fr, path); return fr


def ranking(fr, basis):
    el = fr[fr.eligible.astype(bool)]
    return sorted(el.feature_id.astype(int), key=lambda i: (-fr.loc[i, basis], i))


def tiers(fr, basis):
    return dict(zip(['top', 'middle', 'bottom'], [list(map(int, t)) for t in np.array_split(ranking(fr, basis), 3)]))


def corr_matrix(A):
    C = pd.DataFrame(A).corr().abs().fillna(0).to_numpy(copy=True); np.fill_diagonal(C, 0); return C

## Evaluation engine (one call per experiment/dataset/fold group, checkpointed, artifacts saved)

In [6]:
def run_group(exp, name, cap, fold, conditions, methods, specs):
    """conditions: dicts with cond_id, mechanism, cols, design (tuple for the seed), meta."""
    out_csv = OUT / exp / 'records' / f'{name}_{cap}_{fold}.csv'
    if out_csv.exists(): return pd.read_csv(out_csv)
    A, y, B, z = fold_data(name, cap, fold); key = dkey(name, cap); C = corr_matrix(A)
    rmse_sd = np.maximum(np.std(A, axis=0), 1e-8); minority = int(np.bincount(y).argmin())
    for spec in specs:
        if spec == 'xgb_tuned': tuned_params(name, cap, fold, A.astype(np.float32), y)
    base = {s: baseline(name, cap, fold, s, A, y, B, z) for s in specs}
    rows, arrays = [], {}
    for cond in conditions:
        g = MaskGenerator(cond['mechanism'], RATE, cond['cols']).fit(A, y, C)
        sa = (SEED, key, int(fold), int(MASK_SEED), cond['mechanism'], float(RATE), *cond['design'])
        ma, pa = g.sample(A, y, stable_seed(*sa, 'train')); mb, pb = g.sample(B, z, stable_seed(*sa, 'test'))
        AA, BB = A.copy(), B.copy(); AA[ma] = np.nan; BB[mb] = np.nan
        cid = cond['cond_id']
        arrays[f'{cid}|mask_train_rows'] = np.packbits(ma[:, cond['cols'][0]])
        arrays[f'{cid}|mask_test_rows'] = np.packbits(mb[:, cond['cols'][0]])
        mask_meta = {'realized_train_rate': ma[:, cond['cols'][0]].mean(), 'realized_test_rate': mb[:, cond['cols'][0]].mean(),
                     'expected_test_rate': pb.mean(), 'n_masked_cols': len(cond['cols']),
                     'cols': '|'.join(map(str, cond['cols'])), 'driver': g.driver}
        imp_seed = stable_seed(*sa, 'imputer')
        for method in methods:
            sp = [s for s in specs if method != 'native' or SPEC_NATIVE_OK(s)]
            if not sp: continue
            w0, c0 = time.perf_counter(), time.process_time()
            try:
                comps, info, obj = impute(method, AA, BB, imp_seed)
            except Exception as e:
                rows.append({'exp': exp, 'dataset': name, 'cap': cap, 'fold': fold, 'cond_id': cid, 'imputer': method,
                             'status': f'imputer_failed: {str(e)[:100]}', **cond['meta']}); continue
            imp_wall, imp_cpu = time.perf_counter() - w0, time.process_time() - c0
            recon = np.mean([c[1] for c in comps], axis=0)
            if method != 'native':
                err = ((recon - B) / rmse_sd)[mb]; srmse = float(np.sqrt(np.mean(err ** 2))) if err.size else np.nan
                arrays[f'{cid}|{method}|recon_test'] = recon[mb].astype(np.float32)
            else: srmse = np.nan
            save_models = fold in SAVE_MODEL_FOLDS and exp in SAVE_MODEL_EXPS
            if save_models and obj is not None:
                p_ = ART / 'models' / exp / f'{name}_{cap}_{fold}_{cid}_{method}_imputer.joblib'
                p_.parent.mkdir(parents=True, exist_ok=True); joblib.dump(obj, p_, compress=3)
            for spec in sp:
                w1, c1 = time.perf_counter(), time.process_time(); probs = []; per_imp = []
                seed = spec_seed(spec, key, fold)
                for (ta, tb, ia, ib) in comps:
                    Xa = ta if ia is None else np.concatenate([ta, ia], 1)
                    Xb = tb if ib is None else np.concatenate([tb, ib], 1)
                    P, clf = fit_predict(spec, Xa.astype(np.float32), y, Xb.astype(np.float32), seed, name, cap, fold)
                    probs.append(P); per_imp.append(calculate_metrics(z, P, minority)['macro_f1'])
                P = np.mean(probs, axis=0); met = calculate_metrics(z, P, minority)
                arrays[f'{cid}|{method}|{spec}|proba'] = P.astype(np.float32)
                if save_models and hasattr(clf, 'get_booster') and spec in ('xgb', 'xgb_tuned'):
                    p_ = ART / 'models' / exp / f'{name}_{cap}_{fold}_{cid}_{method}_{spec}.json'
                    clf.get_booster().save_model(str(p_))
                r = {'exp': exp, 'dataset': name, 'cap': cap, 'fold': fold, 'cond_id': cid, 'mechanism': cond['mechanism'],
                     'imputer': method, 'classifier': spec, 'status': 'ok', **cond['meta'], **mask_meta, **met,
                     **{f'baseline_{k}': base[spec][k] for k in METRICS},
                     **{f'drop_{k}': base[spec][k] - met[k] for k in METRICS},
                     'std_rmse': srmse, 'imputer_wall_s': imp_wall, 'imputer_cpu_s': imp_cpu,
                     'clf_wall_s': time.perf_counter() - w1, 'clf_cpu_s': time.process_time() - c1,
                     'n_iter': info['n_iter'], 'max_iter': info['max_iter'], 'converged': info['converged'],
                     'stop_rule': info.get('stop_rule', ''), 'imputer_warning': info['warning'],
                     'between_imputation_sd': float(np.std(per_imp)) if len(per_imp) > 1 else np.nan,
                     'imputer_seed': imp_seed, 'model_seed': seed,
                     'mask_seed_train': stable_seed(*sa, 'train'), 'mask_seed_test': stable_seed(*sa, 'test')}
                r['record_id'] = rid(exp, name, cap, fold, cid, method, spec); rows.append(r)
    df = pd.DataFrame(rows)
    p_ = ART / 'predictions' / exp / f'{name}_{cap}_{fold}.npz'; p_.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(p_, **arrays)
    atomic_csv(df, out_csv)
    return df


def collect(exp):
    fs = sorted((OUT / exp / 'records').glob('*.csv'))
    df = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()
    if len(df): atomic_csv(df, OUT / f'{exp}_records.csv')
    return df


def progress(it, label):
    it = list(it); t0 = time.time()
    for i, x in enumerate(it):
        yield x
        print(f'\r{label}: {i + 1}/{len(it)}  ({(time.time() - t0) / 60:.1f} min)', end='', flush=True)
    print()


CORE6 = ['mean', 'median_indicator', 'knn', 'iterative', 'rf_iterative', 'native']


def single(tier_name, fid, basis, mech, role='original_middle', extra=None):
    """Design tuple reproduces the original seeds when role is the original middle feature on the RF basis."""
    design = (tier_name, 'permutation') if (role == 'original_middle' and basis == 'permutation') else (f'{tier_name}#{fid}', basis)
    return {'cond_id': rid(mech, design, fid)[:10], 'mechanism': mech, 'cols': [int(fid)], 'design': design,
            'meta': {'design': 'single', 'basis': basis, 'tier': tier_name, 'role': role, 'feature_id': int(fid), **(extra or {})}}


def block(kind, cols, basis, mech, extra=None):
    design = (f'block_{kind}', basis)
    return {'cond_id': rid(mech, design, cols)[:10], 'mechanism': mech, 'cols': [int(c) for c in cols], 'design': design,
            'meta': {'design': 'block', 'basis': basis, 'tier': f'block_{kind}', 'role': kind, 'feature_id': -1, **(extra or {})}}

## S1 — Importance stability and model-specific attribution (R1-5, R2-3, R2-5)

In [7]:
if RUN['S1']:
    for name, fold in progress([(n, f) for n in DATASETS for f in FOLD_IDS], 'S1 importance'):
        importance(name, 'cap', fold)
    IMP = pd.concat([importance(n, 'cap', f) for n in DATASETS for f in FOLD_IDS], ignore_index=True)
    atomic_csv(IMP, SUM / 'S1_importance_long.csv')
    rows = []
    for name in DATASETS:
        g = IMP[(IMP.dataset == name) & IMP.eligible.astype(bool)]
        for basis in BASES:
            piv = g.pivot(index='feature_id', columns='fold', values=basis)
            taus = [kendalltau(piv[a], piv[b])[0] for a, b in itertools.combinations(piv.columns, 2)]
            frs = {f: IMP[(IMP.dataset == name) & (IMP.fold == f)].reset_index(drop=True) for f in piv.columns}
            T = {f: tiers(frs[f], basis) for f in piv.columns}
            lab = pd.DataFrame({f: {i: t for t, ids in T[f].items() for i in ids} for f in piv.columns})
            modal = lab.apply(lambda r: r.value_counts().iloc[0] / r.notna().sum(), axis=1).mean()
            tops = [set(T[f]['top']) for f in piv.columns]
            jac = np.mean([len(a & b) / len(a | b) for a, b in itertools.combinations(tops, 2)])
            reps = [T[f]['top'][len(T[f]['top']) // 2] for f in piv.columns]
            vs_rf = np.mean([kendalltau(frs[f].loc[frs[f].eligible.astype(bool), basis],
                                        frs[f].loc[frs[f].eligible.astype(bool), 'permutation'])[0] for f in piv.columns])
            vs_shap = np.mean([kendalltau(frs[f].loc[frs[f].eligible.astype(bool), basis],
                                          frs[f].loc[frs[f].eligible.astype(bool), 'xgb_shap'])[0] for f in piv.columns])
            tier_agree_rf = np.mean([np.mean([({i: t for t, ids in T[f].items() for i in ids}[i] ==
                                               {i: t for t, ids in tiers(frs[f], 'permutation').items() for i in ids}[i])
                                              for i in ranking(frs[f], basis)]) for f in piv.columns])
            rows.append({'dataset': name, 'basis': basis, 'features': len(piv), 'fold_pairwise_kendall_tau': np.nanmean(taus),
                         'tier_modal_agreement': modal, 'top_tier_jaccard': jac,
                         'top_representative_modal_share': pd.Series(reps).value_counts().iloc[0] / len(reps),
                         'share_nonpositive': float((g[basis] <= 0).mean()),
                         'inner_holdout_tau': g.get(f'{basis}_inner_tau', pd.Series([np.nan])).mean(),
                         'tau_vs_rf_permutation_same_fold': vs_rf, 'tau_vs_xgb_shap_same_fold': vs_shap,
                         'tier_agreement_with_rf_permutation': tier_agree_rf})
    S1 = pd.DataFrame(rows); atomic_csv(S1, SUM / 'S1_importance_stability.csv')
    display(S1.round(3))

S1 importance: 42/42  (21.8 min)


,dataset,basis,features,fold_pairwise_kendall_tau,tier_modal_agreement,top_tier_jaccard,top_representative_modal_share,share_nonpositive,inner_holdout_tau,tau_vs_rf_permutation_same_fold,tau_vs_xgb_shap_same_fold,tier_agreement_with_rf_permutation
0,breast_cancer,permutation,30,0.026,0.589,0.278,0.167,0.750,NaN,1.000,0.164,1.000
1,breast_cancer,rf_perm_rep,30,0.182,0.633,0.312,0.333,0.494,0.089,0.175,0.303,0.417
2,breast_cancer,xgb_perm_rep,30,0.337,0.672,0.519,0.167,0.417,0.211,0.123,0.446,0.383
3,breast_cancer,xgb_shap,30,0.620,0.789,0.774,0.333,0.000,0.479,0.164,1.000,0.444
4,banknote,permutation,4,1.000,1.000,1.000,1.000,0.042,NaN,1.000,1.000,1.000
5,banknote,rf_perm_rep,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
6,banknote,xgb_perm_rep,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
7,banknote,xgb_shap,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
8,digits,permutation,61,0.169,0.656,0.382,0.167,0.424,NaN,1.000,0.406,1.000
9,digits,rf_perm_rep,61,0.649,0.806,0.648,0.167,0.292,0.302,0.402,0.621,0.497


## S2 — Tier design: highest vs middle vs tier average; equal-size top/bottom/random/redundancy-matched blocks (R1-6, R2-4, R2-12)

In [8]:
def tier_members(ids, basis):
    n = len(ids); mm = MAX_MEMBERS[basis]
    pos = list(range(n)) if (mm is None or n <= mm) else sorted(
        set(np.linspace(0, n - 1, mm).round().astype(int).tolist() + [n // 2]))
    # position 0 = highest-ranked member; n//2 = the original representative (array_split middle)
    return [(ids[i], 'original_middle' if i == n // 2 else ('highest' if i == 0 else 'member'), i) for i in pos]


def s2_conditions(name, fold):
    fr = importance(name, 'cap', fold); A = fold_data(name, 'cap', fold)[0]; mc = corr_matrix(A).max(axis=1)
    conds = []
    for basis in ['permutation', 'xgb_shap']:
        rk = ranking(fr, basis); T = tiers(fr, basis); rank_of = {f: 1 - i / max(1, len(rk) - 1) for i, f in enumerate(rk)}
        for mech in ['mcar', 'mnar_strong']:
            for tname, ids in T.items():
                for fid, role, pos in tier_members(ids, basis):
                    conds.append(single(tname, fid, basis, mech, role,
                                        {'member_pos': pos, 'tier_size': len(ids), 'importance_rank': rank_of[fid],
                                         'max_abs_corr': mc[fid]}))
            k = max(1, min(3, len(T['top'])))
            top_b = rk[:k]; bot_b = rk[-k:]
            rnd = sorted(np.random.default_rng(stable_seed(SEED, name, fold, MASK_SEED, 'random_block', basis))
                         .choice(rk, k, replace=False).tolist(), key=rk.index)
            cand = [f for f in rk if f not in top_b]; matched = []
            for f in top_b:
                c = min([x for x in cand if x not in matched], key=lambda x: abs(mc[x] - mc[f]))
                matched.append(c)
            matched = sorted(matched, key=rk.index)
            for kind, cols in [('top', top_b), ('bottom', bot_b), ('random', rnd), ('redundancy_matched', matched)]:
                wb = float(np.mean([corr_matrix(A)[i, j] for i, j in itertools.combinations(cols, 2)])) if k > 1 else np.nan
                conds.append(block(kind, cols, basis, mech, {'block_size': k, 'within_block_abs_corr': wb,
                                                             'importance_rank': np.mean([rank_of[c] for c in cols]),
                                                             'max_abs_corr': float(np.mean(mc[cols]))}))
    return conds


if RUN['S2']:
    for name, fold in progress([(n, f) for n in DATASETS for f in FOLD_IDS], 'S2 tier design'):
        run_group('S2', name, 'cap', fold, s2_conditions(name, fold), CORE6, ['xgb'])
    S2 = collect('S2'); print(S2.shape)

S2 tier design: 42/42  (107.3 min)
(15840, 57)


## S3 — Iteration budget, canonical MissForest, MICE with posterior sampling and pooling (R1-9, R1-10, R2-6, R2-7)

In [9]:
S3_METHODS = ['mean', 'native', 'iterative', 'iterative_mi25', 'iterative_allfeat', 'rf_iterative', 'rf_iterative_mi25',
              'rf_iterative_allfeat', 'missforest', 'mice5']


def s3_conditions(name, fold):
    fr = importance(name, 'cap', fold); T = tiers(fr, 'permutation'); rk = ranking(fr, 'permutation')
    top = T['top'][len(T['top']) // 2]; k = min(3, len(rk) - 1); conds = []
    for mech in ['mcar', 'mar_redundant', 'mnar_strong']:
        conds.append(single('top', top, 'permutation', mech, 'original_middle', {'target': 'single'}))
        conds.append(block('top', rk[:k], 'permutation', mech, {'target': f'block{k}', 'block_size': k}))
    return conds


if RUN['S3']:
    for name, fold in progress([(n, f) for n in DATASETS for f in FOLD_IDS], 'S3 iterative'):
        run_group('S3', name, 'cap', fold, s3_conditions(name, fold), S3_METHODS, ['xgb'])
    S3 = collect('S3'); print(S3.shape)

S3 iterative: 42/42  (59.9 min)
(2520, 53)


## S4 — Classifier robustness, tuned XGBoost, XGBoost seed variability (R1-11, R2-8, R2-9)

In [10]:
S4_SPECS = ['lr', 'rf', 'hgb', 'xgb'] + [f'xgb_s{k}' for k in range(1, XGB_SEEDS)] + (['xgb_tuned'] if xgb is not None else [])


def s4_conditions(name, fold):
    fr = importance(name, 'cap', fold); T = tiers(fr, 'permutation'); top = T['top'][len(T['top']) // 2]
    return [single('top', top, 'permutation', m, 'original_middle') for m in ['mcar', 'mar_redundant', 'mnar_strong', 'label']]


if RUN['S4']:
    for name, fold in progress([(n, f) for n in DATASETS for f in FOLD_IDS], 'S4 classifiers'):
        run_group('S4', name, 'cap', fold, s4_conditions(name, fold), CORE6, S4_SPECS)
    S4 = collect('S4'); print(S4.shape)

S4 classifiers: 42/42  (52.9 min)
(8736, 51)


## S5 — 1,500-row cap vs full data (R2-2)

In [11]:
S5_DATASETS = [n for n in DATASETS if 'full' in DATA[n]['versions']]
S5_CAPS = ['cap', 'cap_alt', 'full']
S5_METHODS = ['mean', 'median_indicator', 'knn', 'knn_full', 'iterative', 'rf_iterative', 'native']


def s5_conditions(name, cap, fold):
    fr = importance(name, cap, fold, full=False); T = tiers(fr, 'permutation'); top = T['top'][len(T['top']) // 2]
    return [single('top', top, 'permutation', m, 'original_middle', {'target_feature': fr.loc[top, 'feature']})
            for m in ['mcar', 'mnar_strong']]


if RUN['S5'] and S5_DATASETS:
    dens = []
    for name, cap, fold in progress([(n, c, f) for n in S5_DATASETS for c in S5_CAPS for f in FOLD_IDS], 'S5 row cap'):
        run_group('S5', name, cap, fold, s5_conditions(name, cap, fold), S5_METHODS, ['xgb'])
        A, y, _, _ = fold_data(name, cap, fold); Z = StandardScaler().fit_transform(A)
        d5 = NearestNeighbors(n_neighbors=6).fit(Z).kneighbors(Z)[0][:, 5]
        dens.append({'dataset': name, 'cap': cap, 'fold': fold, 'train_rows': len(A), 'median_5nn_distance': float(np.median(d5)),
                     'minority_train_count': int(np.bincount(y).min()), 'minority_share': float(np.bincount(y).min() / len(y))})
    atomic_csv(pd.DataFrame(dens), SUM / 'S5_neighbourhood_density.csv')
    S5 = collect('S5'); print(S5.shape)

S5 row cap: 72/72  (15.2 min)
(1008, 52)


## S6 — Controlled computational benchmark (R1-18)

In [12]:
def hardware():
    cpu = ''
    try: cpu = next(l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name'))
    except Exception: cpu = platform.processor()
    return {'cpu_model': cpu, 'logical_cpus': os.cpu_count(), 'thread_budget': N_THREADS,
            'ram_gb': round(psutil.virtual_memory().total / 1e9, 1), 'gpu': GPU_INFO, 'platform': platform.platform(), **VERSIONS}


if RUN['S6']:
    atomic_json(hardware(), SUM / 'S6_hardware.json')
    path = SUM / 'S6_compute.csv'
    if not path.exists():
        rows = []; methods = ['mean', 'median_indicator', 'knn', 'iterative', 'iterative_mi25', 'rf_iterative',
                              'rf_iterative_mi25', 'missforest', 'mice5']
        for name in progress(DATASETS, 'S6 compute'):
            A, y, B, z = fold_data(name, 'cap', 0); fr = importance(name, 'cap', 0)
            rk = ranking(fr, 'permutation'); T = tiers(fr, 'permutation')
            for target, cols in [('single', [T['top'][len(T['top']) // 2]]), ('block3', rk[:min(3, len(rk) - 1)])]:
                g = MaskGenerator('mcar', RATE, cols).fit(A, y, corr_matrix(A))
                AA, BB = A.copy(), B.copy(); AA[g.sample(A, y, 1)[0]] = np.nan; BB[g.sample(B, z, 2)[0]] = np.nan
                for m in methods:
                    ws, cs = [], []
                    for rep in range(3):
                        w, c = time.perf_counter(), time.process_time(); impute(m, AA, BB, 7)
                        ws.append(time.perf_counter() - w); cs.append(time.process_time() - c)
                    tracemalloc.start(); rss0 = psutil.Process().memory_info().rss; impute(m, AA, BB, 7)
                    peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
                    rows.append({'dataset': name, 'component': 'imputer', 'name': m, 'target': target, 'rows': len(A),
                                 'features': A.shape[1], 'wall_s_median': np.median(ws), 'cpu_s_median': np.median(cs),
                                 'peak_traced_mb': peak / 1e6, 'rss_delta_mb': (psutil.Process().memory_info().rss - rss0) / 1e6})
            for spec, dev in [('lr', 'cpu'), ('rf', 'cpu'), ('hgb', 'cpu'), ('xgb', 'cuda' if GPU else 'cpu'), ('xgb', 'cpu')]:
                ws, cs = [], []
                for rep in range(3):
                    w, c = time.perf_counter(), time.process_time()
                    clf = make_clf(spec, 1, len(np.unique(y)), device=dev); clf.fit(A.astype(np.float32), y)
                    clf.predict_proba(B.astype(np.float32)); ws.append(time.perf_counter() - w); cs.append(time.process_time() - c)
                tracemalloc.start(); make_clf(spec, 1, len(np.unique(y)), device=dev).fit(A.astype(np.float32), y)
                peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
                rows.append({'dataset': name, 'component': 'classifier', 'name': f'{spec}_{dev}', 'target': 'complete',
                             'rows': len(A), 'features': A.shape[1], 'wall_s_median': np.median(ws), 'cpu_s_median': np.median(cs),
                             'peak_traced_mb': peak / 1e6, 'rss_delta_mb': np.nan})
        atomic_csv(pd.DataFrame(rows), path)
    S6 = pd.read_csv(path)
    display(S6.groupby(['component', 'name', 'target'])[['wall_s_median', 'cpu_s_median', 'peak_traced_mb']].median().round(3))

S6 compute: 7/7  (16.6 min)


wall_s_median  cpu_s_median  \
component  name              target                                  
classifier hgb_cpu           complete          0.158         0.612   
           lr_cpu            complete          0.007         0.029   
           rf_cpu            complete          0.320         0.590   
           xgb_cpu           complete          0.085         0.336   
           xgb_cuda          complete          0.181         0.289   
imputer    iterative         block3            0.041         0.147   
                             single            0.016         0.052   
           iterative_mi25    block3            0.176         0.177   
                             single            0.062         0.062   
           knn               block3            0.026         0.077   
                             single            0.017         0.051   
           mean              block3            0.002         0.002   
                             single            0.002         0.007   
           median_indicator  block3            0.002         0.002   
                             single            0.002         0.002   
           mice5             block3            1.009         1.010   
                             single            0.345         0.345   
           missforest        block3            3.025         6.327   
                             single            0.777         1.765   
           rf_iterative      block3            1.447         2.577   
                             single            0.488         0.784   
           rf_iterative_mi25 block3            7.174        12.614   
                             single            2.289         3.816   

                                       peak_traced_mb  
component  name              target                    
classifier hgb_cpu           complete           1.004  
           lr_cpu            complete           0.284  
           rf_cpu            complete           0.533  
           xgb_cpu           complete           0.054  
           xgb_cuda          complete           0.054  
imputer    iterative         block3             0.669  
                             single             0.654  
           iterative_mi25    block3             0.719  
                             single             0.654  
           knn               block3             5.642  
                             single             4.418  
           mean              block3             0.469  
                             single             0.469  
           median_indicator  block3             0.664  
                             single             0.664  
           mice5             block3             3.180  
                             single             2.124  
           missforest        block3             2.407  
                             single             0.992  
           rf_iterative      block3             1.494  
                             single             0.998  
           rf_iterative_mi25 block3             4.624  
                             single             2.079

## S7 — Reproducibility: re-run check from archived artifacts, comparison with the original run, RF-imputer seed sensitivity (R1-17, R2-10)

In [13]:
def rerun_record(r):
    """Rebuild one evaluation from archived split indices + masks and the stored seeds."""
    sp = np.load(ART / 'splits' / f"{r.dataset}_{r.cap}.npz"); tr, te = sp[f'train_{r.fold}'], sp[f'test_{r.fold}']
    v = DATA[r.dataset]['versions'][r.cap]; A, y, B, z = v['X'][tr], v['y'][tr], v['X'][te], v['y'][te]
    arr = np.load(ART / 'predictions' / r.exp / f"{r.dataset}_{r.cap}_{r.fold}.npz")
    cols = [int(float(c)) for c in str(r.cols).split('|')]
    ra = np.unpackbits(arr[f'{r.cond_id}|mask_train_rows'])[:len(A)].astype(bool)
    rb = np.unpackbits(arr[f'{r.cond_id}|mask_test_rows'])[:len(B)].astype(bool)
    AA, BB = A.copy(), B.copy(); AA[np.ix_(ra, cols)] = np.nan; BB[np.ix_(rb, cols)] = np.nan
    comps, _, _ = impute(r.imputer, AA, BB, int(r.imputer_seed)); probs = []
    for (ta, tb, ia, ib) in comps:
        Xa = ta if ia is None else np.concatenate([ta, ia], 1); Xb = tb if ib is None else np.concatenate([tb, ib], 1)
        probs.append(fit_predict(r.classifier, Xa.astype(np.float32), y, Xb.astype(np.float32), int(r.model_seed),
                                 r.dataset, r.cap, int(r.fold))[0])
    P = np.mean(probs, 0); stored = arr[f'{r.cond_id}|{r.imputer}|{r.classifier}|proba']
    return calculate_metrics(z, P, int(np.bincount(y).argmin()))['macro_f1'], float(np.abs(P - stored).max())


def find_original():
    if ORIGINAL_RESULTS: return Path(ORIGINAL_RESULTS)
    for p in ATTACHED:
        if p.name == 'results.csv' and (p.parent / 'plan.csv').exists(): return p.parent
    return None


if RUN['S7']:
    pool = pd.concat([collect(e) for e in ['S2', 'S3', 'S4', 'S5'] if (OUT / e).exists()], ignore_index=True)
    pool = pool[pool.status == 'ok']
    samp = pool.sample(min(REPRO_SAMPLE, len(pool)), random_state=1)
    rr = []
    for _, r in progress(list(samp.iterrows()), 'S7 re-run'):
        f1, dp = rerun_record(r)
        rr.append({'record_id': r.record_id, 'exp': r.exp, 'dataset': r.dataset, 'imputer': r.imputer, 'classifier': r.classifier,
                   'stored_macro_f1': r.macro_f1, 'rerun_macro_f1': f1, 'abs_diff': abs(f1 - r.macro_f1), 'max_proba_diff': dp})
    RR = pd.DataFrame(rr); atomic_csv(RR, SUM / 'S7_rerun_check.csv')
    print('Re-run identical macro-F1:', (RR.abs_diff < 1e-12).mean(), '| max prob diff:', RR.max_proba_diff.max())

    orig = find_original()
    if orig is not None:
        o = pd.read_csv(orig / 'results.csv')
        o = o[(o.stage == 'core') & (o.mask_seed == MASK_SEED) & (o.rate == RATE) & (o.importance_basis == 'permutation')]
        new = pool[(pool.classifier == 'xgb') & (pool.role == 'original_middle') & (pool.basis == 'permutation') & (pool.cap == 'cap')]
        m = new.merge(o[['dataset', 'fold', 'mechanism', 'tier', 'imputer', 'macro_f1']], on=['dataset', 'fold', 'mechanism', 'tier', 'imputer'],
                      suffixes=('', '_original'))
        m['abs_diff'] = (m.macro_f1 - m.macro_f1_original).abs()
        atomic_csv(m[['exp', 'dataset', 'fold', 'mechanism', 'tier', 'imputer', 'macro_f1', 'macro_f1_original', 'abs_diff']],
                   SUM / 'S7_match_original.csv')
        print(f'Matched {len(m)} records to the original run; identical: {(m.abs_diff < 1e-9).mean():.1%}')
    else:
        print('Original results not attached - skipping comparison with the original run.')

    # RF-iterative / MissForest seed sensitivity, incl. the Spambase strong-MNAR fold-1 case flagged by both reviewers
    path = SUM / 'S7_imputer_seed_sensitivity.csv'
    if not path.exists():
        rows = []
        cases = [(n, FOLD_IDS[0], 'mcar') for n in DATASETS]
        if 'spambase' in DATASETS and 1 in FOLD_IDS: cases.append(('spambase', 1, 'mnar_strong'))
        for name, fold, mech in progress(cases, 'S7 seed sensitivity'):
            A, y, B, z = fold_data(name, 'cap', fold); fr = importance(name, 'cap', fold); T = tiers(fr, 'permutation')
            cond = single('top', T['top'][len(T['top']) // 2], 'permutation', mech)
            g = MaskGenerator(mech, RATE, cond['cols']).fit(A, y, corr_matrix(A))
            sa = (SEED, name, int(fold), int(MASK_SEED), mech, float(RATE), *cond['design'])
            AA, BB = A.copy(), B.copy(); AA[g.sample(A, y, stable_seed(*sa, 'train'))[0]] = np.nan
            BB[g.sample(B, z, stable_seed(*sa, 'test'))[0]] = np.nan
            for method in ['rf_iterative', 'missforest', 'knn']:
                for s in range(SEED_SENSITIVITY_SEEDS):
                    comps, _, _ = impute(method, AA, BB, stable_seed(SEED, 'seed_sensitivity', s))
                    ta, tb, _, _ = comps[0]
                    P, _ = fit_predict('xgb', ta.astype(np.float32), y, tb.astype(np.float32), spec_seed('xgb', name, fold), name, 'cap', fold)
                    rows.append({'dataset': name, 'fold': fold, 'mechanism': mech, 'imputer': method, 'imputer_seed_index': s,
                                 'macro_f1': calculate_metrics(z, P, int(np.bincount(y).argmin()))['macro_f1']})
        atomic_csv(pd.DataFrame(rows), path)

    # Thread-order test: same seeds, same masks, repeated runs; original multi-threaded forest prediction vs serial
    path = SUM / 'S7_thread_order_test.csv'
    if not path.exists():
        rows = []; keep_flag = DETERMINISTIC_PREDICT
        cases = [(n, FOLD_IDS[0], 'mcar') for n in DATASETS]
        if 'spambase' in DATASETS and 1 in FOLD_IDS: cases.insert(0, ('spambase', 1, 'mnar_strong'))
        for name, fold, mech in progress(cases, 'S7 thread-order test'):
            A, y, B, z = fold_data(name, 'cap', fold); fr = importance(name, 'cap', fold); T = tiers(fr, 'permutation')
            cond = single('top', T['top'][len(T['top']) // 2], 'permutation', mech)
            g = MaskGenerator(mech, RATE, cond['cols']).fit(A, y, corr_matrix(A))
            sa = (SEED, name, int(fold), int(MASK_SEED), mech, float(RATE), *cond['design'])
            AA, BB = A.copy(), B.copy(); AA[g.sample(A, y, stable_seed(*sa, 'train'))[0]] = np.nan
            BB[g.sample(B, z, stable_seed(*sa, 'test'))[0]] = np.nan
            for mode in [False, True]:
                DETERMINISTIC_PREDICT = mode; ref = None
                for rep_ in range(THREAD_REPEATS):
                    comps, _, _ = impute('rf_iterative', AA, BB, stable_seed(*sa, 'imputer'))   # original imputer seed
                    ta, tb, _, _ = comps[0]
                    P, _ = fit_predict('xgb', ta.astype(np.float32), y, tb.astype(np.float32), spec_seed('xgb', name, fold),
                                       name, 'cap', fold)
                    ref = tb if ref is None else ref
                    rows.append({'dataset': name, 'fold': fold, 'mechanism': mech,
                                 'forest_prediction': 'serial (deterministic)' if mode else 'multi-threaded (original)',
                                 'repeat': rep_, 'macro_f1': calculate_metrics(z, P, int(np.bincount(y).argmin()))['macro_f1'],
                                 'max_abs_imputed_diff_vs_first_run': float(np.nanmax(np.abs(tb - ref)))})
        DETERMINISTIC_PREDICT = keep_flag
        atomic_csv(pd.DataFrame(rows), path)
    TT = pd.read_csv(path)
    TT_SUM = TT.groupby(['dataset', 'fold', 'mechanism', 'forest_prediction']).agg(
        repeats=('macro_f1', 'size'), distinct_macro_f1=('macro_f1', lambda v: v.round(12).nunique()),
        min_macro_f1=('macro_f1', 'min'), max_macro_f1=('macro_f1', 'max'),
        max_imputed_diff=('max_abs_imputed_diff_vs_first_run', 'max')).reset_index()
    display(TT_SUM)
    SS = pd.read_csv(SUM / 'S7_imputer_seed_sensitivity.csv')
    display(SS.groupby(['dataset', 'fold', 'mechanism', 'imputer']).macro_f1.agg(['mean', 'std', 'min', 'max']).mul(100).round(3))

S7 re-run: 40/40  (0.4 min)
Re-run identical macro-F1: 1.0 | max prob diff: 2.9795782174879548e-08
Original results not attached - skipping comparison with the original run.
S7 seed sensitivity: 8/8  (2.8 min)
S7 thread-order test: 8/8  (1.7 min)


,dataset,fold,mechanism,forest_prediction,repeats,distinct_macro_f1,min_macro_f1,max_macro_f1,max_imputed_diff
0,banknote,0,mcar,multi-threaded (original),8,1,0.993272,0.993272,3.552714e-15
1,banknote,0,mcar,serial (deterministic),8,1,0.993272,0.993272,0.000000e+00
2,breast_cancer,0,mcar,multi-threaded (original),8,1,0.965646,0.965646,1.387779e-17
3,breast_cancer,0,mcar,serial (deterministic),8,1,0.965646,0.965646,0.000000e+00
4,digits,0,mcar,multi-threaded (original),8,1,0.966136,0.966136,3.552714e-15
5,digits,0,mcar,serial (deterministic),8,1,0.966136,0.966136,0.000000e+00
6,ionosphere,0,mcar,multi-threaded (original),8,1,0.942308,0.942308,2.220446e-16
7,ionosphere,0,mcar,serial (deterministic),8,1,0.942308,0.942308,0.000000e+00
8,magic,0,mcar,multi-threaded (original),8,1,0.843122,0.843122,8.881784e-16
9,magic,0,mcar,serial (deterministic),8,1,0.843122,0.843122,0.000000e+00


mean    std     min     max
dataset       fold mechanism   imputer                                    
banknote      0    mcar        knn           99.328  0.000  99.328  99.328
                               missforest    99.238  0.217  98.879  99.552
                               rf_iterative  99.238  0.116  99.103  99.328
breast_cancer 0    mcar        knn           97.146  0.000  97.146  97.146
                               missforest    97.146  0.000  97.146  97.146
                               rf_iterative  97.088  0.184  96.565  97.146
digits        0    mcar        knn           97.006  0.000  97.006  97.006
                               missforest    96.114  0.273  95.407  96.411
                               rf_iterative  96.375  0.349  95.811  96.815
ionosphere    0    mcar        knn           94.231  0.000  94.231  94.231
                               missforest    94.146  0.538  93.312  95.223
                               rf_iterative  93.654  0.811  92.308  94.231
magic         0    mcar        knn           84.891  0.000  84.891  84.891
                               missforest    84.264  0.484  83.216  84.760
                               rf_iterative  83.857  0.407  83.265  84.514
phoneme       0    mcar        knn           84.038  0.000  84.038  84.038
                               missforest    83.500  0.457  82.652  84.183
                               rf_iterative  83.126  0.726  82.371  84.329
spambase      0    mcar        knn           91.528  0.000  91.528  91.528
                               missforest    91.707  0.356  91.122  92.171
                               rf_iterative  91.742  0.261  91.342  92.171
              1    mnar_strong knn           93.536  0.000  93.536  93.536
                               missforest    93.235  0.226  92.917  93.547
                               rf_iterative  93.134  0.263  92.500  93.355

## S8 — Archive of the ORIGINAL run: split indices and every missingness mask, regenerated from the original seeds (R1-17, R2-10)
No models are fitted. Masks of the core grid (8 mechanisms, rates, tiers, blocks, mask seeds 17 and 47) and of the
train/test mismatch grid are rebuilt with the original seed scheme and compared with the original mask_checks.csv.

In [14]:
ORIG_MECHS = ['mcar', 'mar_redundant', 'mar_unrelated', 'mnar_mild', 'mnar_moderate', 'mnar_strong', 'mnar_censor', 'label']
ORIG_MASK_SEEDS = [17, 47]
MISMATCH_SCENARIOS = {'matched': ('mcar', .2, 'mcar', .2), 'train_only': ('mcar', .2, 'mcar', 0.),
                      'test_only': ('mcar', 0., 'mcar', .2), 'rate_shift': ('mcar', .1, 'mcar', .4),
                      'mechanism_shift': ('mcar', .2, 'mnar_strong', .2), 'matched_mnar': ('mnar_strong', .2, 'mnar_strong', .2),
                      'matched_label': ('label', .2, 'label', .2)}


def original_grid():
    g = []
    for m in ORIG_MECHS:
        for r in ([.1, .2, .4] if m in ('mcar', 'mnar_strong', 'label') else [.2]):
            g += [(m, r, t) for t in ['top', 'middle', 'bottom']]
    return g + [(m, .2, 'block') for m in ['block_top', 'block_random']]


if RUN['S8']:
    idx_path = SUM / 'S8_original_mask_index.csv'
    if not idx_path.exists():
        index = []
        mismatch_sets = [n for n in ORDER if n in DATA][:5]
        for name in progress(DATASETS, 'S8 original masks'):
            arrays = {}
            for fold in FOLD_IDS:
                A, y, B, z = fold_data(name, 'cap', fold); fr = importance(name, 'cap', fold, full=False)
                rk = ranking(fr, 'permutation'); T = tiers(fr, 'permutation'); C = corr_matrix(A)
                jobs = []
                for ms in ORIG_MASK_SEEDS:
                    for m, r, t in original_grid():
                        if t == 'block':
                            k = min(3, max(2, len(rk) // 4))
                            cols = rk[:k] if m == 'block_top' else np.random.default_rng(
                                stable_seed(SEED, name, fold, ms, 'random_block')).choice(rk, k, replace=False).tolist()
                        else:
                            cols = [T[t][len(T[t]) // 2]]
                        sa = (SEED, name, int(fold), int(ms), m, float(r), t, 'permutation')
                        jobs.append(('core', '', ms, (m, r), (m, r), cols, t, stable_seed(*sa, 'train'), stable_seed(*sa, 'test')))
                if name in mismatch_sets and fold < FOLDS:
                    j = T['top'][len(T['top']) // 2]
                    for sc, (mt, rt, me, re_) in MISMATCH_SCENARIOS.items():
                        jobs.append(('mismatch', sc, 17, (mt, rt), (me, re_), [j], 'top',
                                     stable_seed(SEED, name, fold, 17, mt, rt, 'top', 'permutation', 'train'),
                                     stable_seed(SEED, name, fold, 17, me, re_, 'top', 'permutation', 'test')))
                for stage, sc, ms, (mt, rt), (me, re_), cols, t, s_tr, s_te in jobs:
                    gt = MaskGenerator(mt, rt, cols).fit(A, y, C); ge = MaskGenerator(me, re_, cols).fit(A, y, C)
                    ma = gt.sample(A, y, s_tr)[0][:, cols[0]]; mb = ge.sample(B, z, s_te)[0][:, cols[0]]
                    key = f'{stage}|{sc}|{fold}|{ms}|{mt}|{rt}|{me}|{re_}|{t}'
                    arrays[key + '|train'] = np.packbits(ma); arrays[key + '|test'] = np.packbits(mb)
                    for part, mm, mech, rate in [('train', ma, mt, rt), ('test', mb, me, re_)]:
                        index.append({'stage': stage, 'scenario': sc, 'dataset': name, 'fold': fold, 'mask_seed': ms,
                                      'mechanism': mech, 'rate': rate, 'tier': t, 'partition': part,
                                      'cols': '|'.join(map(str, cols)), 'realized_target_rate': float(mm.mean()),
                                      'array_key': key + '|' + part})
            pth = ART / 'original_run' / f'masks_{name}.npz'; pth.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(pth, **arrays)
        atomic_csv(pd.DataFrame(index), idx_path)
    S8 = pd.read_csv(idx_path)
    orig = find_original(); match_rows = []
    if orig is not None:
        mc = pd.read_csv(orig / 'mask_checks.csv')
        mc = mc[(mc.stage == 'core') & (mc.importance_basis == 'permutation')]
        a = S8[S8.stage == 'core'].copy()
        mrg = a.merge(mc[['dataset', 'fold', 'mask_seed', 'mechanism', 'rate', 'tier', 'partition', 'realized_target_rate']],
                      on=['dataset', 'fold', 'mask_seed', 'mechanism', 'rate', 'tier', 'partition'], suffixes=('', '_original'))
        match_rows.append({'grid': 'core', 'masks_compared': len(mrg),
                           'identical_realized_rate': float((mrg.realized_target_rate - mrg.realized_target_rate_original).abs().lt(1e-12).mean())})
        if (orig / 'mismatch_mask_checks.csv').exists():
            mm = pd.read_csv(orig / 'mismatch_mask_checks.csv')
            b = S8[S8.stage == 'mismatch']
            mrg2 = b.merge(mm[['dataset', 'fold', 'scenario', 'partition', 'realized_target_rate']],
                           on=['dataset', 'fold', 'scenario', 'partition'], suffixes=('', '_original'))
            match_rows.append({'grid': 'mismatch', 'masks_compared': len(mrg2),
                               'identical_realized_rate': float((mrg2.realized_target_rate - mrg2.realized_target_rate_original).abs().lt(1e-12).mean())})
    else:
        print('Original results not attached - masks archived but not compared with the original mask_checks.csv.')
    S8_MATCH = pd.DataFrame(match_rows); atomic_csv(S8_MATCH, SUM / 'S8_mask_match.csv')
    print(f'Archived {S8.array_key.nunique()} original masks for {S8.dataset.nunique()} datasets'); display(S8_MATCH)

S8 original masks: 7/7  (0.2 min)
Original results not attached - masks archived but not compared with the original mask_checks.csv.
Archived 1098 original masks for 7 datasets


""


## Summary tables for the response letter (all drops in macro-F1 percentage points; folds averaged within dataset first)

In [15]:
rng = np.random.default_rng(SEED)


def paired(a, b):
    d = (a - b).dropna()
    if len(d) < 2: return {'n_datasets': len(d), 'mean_diff_pp': d.mean() * 100 if len(d) else np.nan}
    bs = [rng.choice(d.values, len(d)).mean() for _ in range(BOOTSTRAPS)]
    try: p = wilcoxon(d.values).pvalue if np.any(d.values != 0) else 1.
    except ValueError: p = np.nan
    return {'n_datasets': len(d), 'mean_diff_pp': d.mean() * 100, 'ci_low_pp': np.percentile(bs, 2.5) * 100,
            'ci_high_pp': np.percentile(bs, 97.5) * 100, 'wilcoxon_p': p, 'datasets_positive': int((d > 0).sum())}


def by_ds(df, cols, val='drop_macro_f1'):
    return df.groupby(cols + ['dataset'])[val].mean()


LINES = ['# Supplementary results - numbers for the response letter', '']


def section(title, df, comments):
    LINES.append(f'## {title} ({comments})')
    try: txt = df.to_markdown(index=False, floatfmt='.3f')
    except ImportError: txt = '```\n' + df.to_string(index=False) + '\n```'
    LINES.append(txt if len(df) else '_no data_')
    LINES.append(''); display(df)


if RUN['S1']:
    section('Importance stability', S1.round(3), 'R1-5, R2-3, R2-5')

if RUN['S2'] and len(S2):
    s = S2[(S2.status == 'ok') & (S2.design == 'single')]
    rows = []
    for (basis, mech, tier), g in s.groupby(['basis', 'mechanism', 'tier']):
        orig = by_ds(g[g.role == 'original_middle'], ['imputer']).groupby('dataset').mean()
        high = by_ds(g[g.member_pos == 0], ['imputer']).groupby('dataset').mean()
        avg = g.groupby(['dataset', 'imputer', 'feature_id']).drop_macro_f1.mean().groupby(['dataset', 'imputer']).mean()
        avg_ds = avg.groupby('dataset').mean()
        im_o = by_ds(g[g.role == 'original_middle'], ['imputer']).groupby('imputer').mean()
        im_a = avg.groupby('imputer').mean()
        rho = spearmanr(im_o.reindex(im_a.index), im_a)[0] if len(im_a) > 2 else np.nan
        rows.append({'basis': basis, 'mechanism': mech, 'tier': tier, 'drop_original_middle_pp': orig.mean() * 100,
                     'drop_highest_pp': high.mean() * 100, 'drop_tier_average_pp': avg_ds.mean() * 100,
                     'imputer_rank_spearman_middle_vs_average': rho,
                     'same_best_imputer': im_o.idxmin() == im_a.idxmin() if len(im_o) else np.nan,
                     **{f'highest_minus_middle_{k}': v for k, v in paired(high, orig).items()}})
    section('Tier representative design', pd.DataFrame(rows), 'R1-6, R2-4')
    b = S2[(S2.status == 'ok') & (S2.design == 'block')]
    rows = []
    for (basis, mech), g in b.groupby(['basis', 'mechanism']):
        d = {k: by_ds(g[g.role == k], ['imputer']).groupby('dataset').mean() for k in ['top', 'bottom', 'random', 'redundancy_matched']}
        row = {'basis': basis, 'mechanism': mech, **{f'drop_{k}_pp': v.mean() * 100 for k, v in d.items()},
               **{f'redund_{k}': g[g.role == k].max_abs_corr.mean() for k in d},
               **{f'within_corr_{k}': g[g.role == k].within_block_abs_corr.mean() for k in d},
               **{f'mean_rank_{k}': g[g.role == k].importance_rank.mean() for k in d}}
        for k in ['bottom', 'random', 'redundancy_matched']:
            pr = paired(d['top'], d[k]); row[f'top_minus_{k}_pp'] = pr.get('mean_diff_pp'); row[f'top_minus_{k}_p'] = pr.get('wilcoxon_p')
            row[f'top_minus_{k}_ci'] = f"[{pr.get('ci_low_pp', np.nan):.2f}, {pr.get('ci_high_pp', np.nan):.2f}]"
        rows.append(row)
    section('Equal-size block controls', pd.DataFrame(rows), 'R2-12')

if RUN['S3'] and len(S3):
    s = S3[S3.status == 'ok']
    conv = s.groupby(['imputer', 'target']).agg(converged_share=('converged', lambda x: pd.to_numeric(x, errors='coerce').mean()),
                                                median_iter=('n_iter', 'median'),
                                                warning_share=('imputer_warning', lambda x: x.fillna('').astype(str).str.len().gt(0).mean()),
                                                drop_pp=('drop_macro_f1', lambda x: x.mean() * 100),
                                                std_rmse=('std_rmse', 'mean'), imputer_s=('imputer_wall_s', 'median')).reset_index()
    section('Imputer convergence and performance', conv, 'R1-9, R2-6, R2-7, R1-10')
    rows = []
    for (target, mech), g in s.groupby(['target', 'mechanism']):
        d = by_ds(g, ['imputer']).unstack('imputer')
        for a, bb in [('iterative_mi25', 'iterative'), ('rf_iterative_mi25', 'rf_iterative'), ('iterative_allfeat', 'iterative'),
                      ('rf_iterative_allfeat', 'rf_iterative'), ('missforest', 'rf_iterative'),
                      ('mice5', 'iterative'), ('missforest', 'mean'), ('mice5', 'mean')]:
            if a in d and bb in d:
                rows.append({'target': target, 'mechanism': mech, 'contrast': f'{a} - {bb}', **paired(d[a], d[bb]),
                             'max_abs_dataset_diff_pp': (d[a] - d[bb]).abs().max() * 100})
    section('Paired contrasts (drop difference; negative = better)', pd.DataFrame(rows), 'R1-9, R2-6, R2-7')

if RUN['S4'] and len(S4):
    s = S4[S4.status == 'ok'].copy(); s['clf'] = s.classifier.where(~s.classifier.str.startswith('xgb_s'), 'xgb')
    tab = s.groupby(['clf', 'imputer', 'dataset']).drop_macro_f1.mean().groupby(['clf', 'imputer']).mean().unstack('clf') * 100
    section('Mean drop by classifier x imputer (pp)', tab.reset_index().round(3), 'R1-11, R2-8')
    ref = tab['xgb']; rows = []
    for c in tab:
        common = tab[c].dropna().index.intersection(ref.dropna().index)
        rows.append({'classifier': c, 'spearman_vs_xgb_imputer_order': spearmanr(tab.loc[common, c], ref[common])[0],
                     'best_imputer': tab[c].idxmin(), 'baseline_macro_f1': s[s.clf == c].baseline_macro_f1.mean()})
    section('Imputer-ordering agreement across classifiers', pd.DataFrame(rows), 'R1-11, R2-8')
    if 'xgb_tuned' in s.classifier.unique():
        t = by_ds(s[s.classifier == 'xgb_tuned'], ['imputer']).unstack('imputer'); dft = by_ds(s[s.classifier == 'xgb'], ['imputer']).unstack('imputer')
        section('Tuned minus default XGBoost (drop)', pd.DataFrame([{'imputer': i, **paired(t[i], dft[i])} for i in t]), 'R2-8')
    x = s[s.classifier.str.match(r'^xgb(_s\d+)?$')]
    seed_sd = x.groupby(['dataset', 'fold', 'mechanism', 'imputer']).drop_macro_f1.std()
    mean_by = x.groupby(['dataset', 'fold', 'mechanism', 'imputer']).drop_macro_f1.mean()
    between = mean_by.groupby(['dataset', 'fold', 'mechanism']).std()
    win = x.loc[x.groupby(['dataset', 'fold', 'mechanism', 'classifier']).drop_macro_f1.idxmin()]
    ref_win = mean_by.groupby(['dataset', 'fold', 'mechanism']).idxmin().map(lambda t: t[-1])
    win['ref'] = win.set_index(['dataset', 'fold', 'mechanism']).index.map(ref_win)
    seed_tab = pd.DataFrame({'xgb_seeds': [x.classifier.nunique()], 'median_seed_sd_pp': [seed_sd.median() * 100],
                             'median_between_imputer_sd_pp': [between.median() * 100],
                             'winner_same_as_seed_average_share': [(win.imputer == win.ref).mean()]})
    section('XGBoost seed variability vs between-imputer differences', seed_tab, 'R2-9')

if RUN['S5'] and S5_DATASETS and len(S5):
    s = S5[S5.status == 'ok']
    t = s.groupby(['dataset', 'cap', 'imputer']).agg(drop_pp=('drop_macro_f1', lambda v: v.mean() * 100),
                                                     minority_drop_pp=('drop_minority_f1', lambda v: v.mean() * 100),
                                                     baseline_f1=('baseline_macro_f1', 'mean')).reset_index()
    section('Capped vs full data', t, 'R2-2')
    rows = []
    for name, g in t.groupby('dataset'):
        p_ = g.pivot(index='imputer', columns='cap', values='drop_pp').dropna()
        rows.append({'dataset': name, 'imputer_order_spearman_cap_vs_full': spearmanr(p_['cap'], p_['full'])[0],
                     'imputer_order_spearman_cap_vs_second_subsample': spearmanr(p_['cap'], p_['cap_alt'])[0],
                     'best_cap': p_['cap'].idxmin(), 'best_second_subsample': p_['cap_alt'].idxmin(),
                     'best_full': p_['full'].idxmin(),
                     'mean_abs_drop_change_cap_vs_full_pp': (p_['cap'] - p_['full']).abs().mean(),
                     'mean_abs_drop_change_cap_vs_second_subsample_pp': (p_['cap'] - p_['cap_alt']).abs().mean()})
    section('Imputer ordering, capped vs full', pd.DataFrame(rows), 'R2-2')
    section('Neighbourhood density', pd.read_csv(SUM / 'S5_neighbourhood_density.csv').groupby(['dataset', 'cap']).mean(numeric_only=True)
            .reset_index().round(3), 'R2-2')

if RUN['S6']:
    section('Compute (fixed threads)', S6.groupby(['component', 'name', 'target'])[['wall_s_median', 'cpu_s_median', 'peak_traced_mb']]
            .median().reset_index().round(3), 'R1-18')
if RUN['S7']:
    section('Re-run from archived artifacts', RR.round(6), 'R1-17, R2-10')
    section('Imputer seed sensitivity (macro-F1, pp)', SS.groupby(['dataset', 'fold', 'mechanism', 'imputer']).macro_f1
            .agg(['mean', 'std', 'min', 'max']).mul(100).reset_index().round(3), 'R2-10, R2-9')
if RUN['S7']:
    section('Thread-order test on identical seeds and masks (RF-iterative + XGBoost)', TT_SUM, 'R2-10')
if RUN['S8']:
    section('Original-run masks regenerated and checked', S8_MATCH if len(S8_MATCH) else
            pd.DataFrame([{'archived_masks': S8.array_key.nunique(), 'note': 'original results not attached'}]), 'R1-17, R2-10')
(SUM / 'response_numbers.md').write_text('\n'.join(LINES))
print('Wrote', SUM / 'response_numbers.md')

,dataset,basis,features,fold_pairwise_kendall_tau,tier_modal_agreement,top_tier_jaccard,top_representative_modal_share,share_nonpositive,inner_holdout_tau,tau_vs_rf_permutation_same_fold,tau_vs_xgb_shap_same_fold,tier_agreement_with_rf_permutation
0,breast_cancer,permutation,30,0.026,0.589,0.278,0.167,0.750,NaN,1.000,0.164,1.000
1,breast_cancer,rf_perm_rep,30,0.182,0.633,0.312,0.333,0.494,0.089,0.175,0.303,0.417
2,breast_cancer,xgb_perm_rep,30,0.337,0.672,0.519,0.167,0.417,0.211,0.123,0.446,0.383
3,breast_cancer,xgb_shap,30,0.620,0.789,0.774,0.333,0.000,0.479,0.164,1.000,0.444
4,banknote,permutation,4,1.000,1.000,1.000,1.000,0.042,NaN,1.000,1.000,1.000
5,banknote,rf_perm_rep,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
6,banknote,xgb_perm_rep,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
7,banknote,xgb_shap,4,1.000,1.000,1.000,1.000,0.000,1.000,1.000,1.000,1.000
8,digits,permutation,61,0.169,0.656,0.382,0.167,0.424,NaN,1.000,0.406,1.000
9,digits,rf_perm_rep,61,0.649,0.806,0.648,0.167,0.292,0.302,0.402,0.621,0.497


,basis,mechanism,tier,drop_original_middle_pp,drop_highest_pp,drop_tier_average_pp,imputer_rank_spearman_middle_vs_average,same_best_imputer,highest_minus_middle_n_datasets,highest_minus_middle_mean_diff_pp,highest_minus_middle_ci_low_pp,highest_minus_middle_ci_high_pp,highest_minus_middle_wilcoxon_p,highest_minus_middle_datasets_positive
0,permutation,mcar,bottom,0.203275,0.135894,0.194264,0.885714,True,7,-0.067381,-0.137245,-0.007302,0.187500,1
1,permutation,mcar,middle,0.470515,0.609125,0.508133,0.485714,True,7,0.138610,0.026394,0.292195,0.062500,5
2,permutation,mcar,top,0.527511,0.768337,0.529062,0.771429,True,7,0.240825,-0.143588,0.645362,0.468750,4
3,permutation,mnar_strong,bottom,0.358668,0.387519,0.366347,0.828571,True,7,0.028851,-0.013224,0.075508,0.625000,3
4,permutation,mnar_strong,middle,0.420038,0.446390,0.436588,0.942857,True,7,0.026352,-0.098284,0.180774,1.000000,3
5,permutation,mnar_strong,top,0.274205,0.787066,0.430275,0.714286,True,7,0.512861,0.037019,1.132667,0.296875,4
6,xgb_shap,mcar,bottom,0.207132,0.205964,0.216770,0.771429,True,7,-0.001168,-0.064292,0.076895,0.812500,2
7,xgb_shap,mcar,middle,0.434694,0.480546,0.481071,0.542857,False,7,0.045852,-0.155656,0.294210,1.000000,2
8,xgb_shap,mcar,top,0.471713,0.814227,0.654458,0.885714,True,7,0.342514,-0.110285,0.926330,0.578125,4
9,xgb_shap,mnar_strong,bottom,0.226355,0.196983,0.220262,0.885714,False,7,-0.029372,-0.080466,0.008916,0.312500,2


,basis,mechanism,drop_top_pp,drop_bottom_pp,drop_random_pp,drop_redundancy_matched_pp,redund_top,redund_bottom,redund_random,redund_redundancy_matched,...,mean_rank_redundancy_matched,top_minus_bottom_pp,top_minus_bottom_p,top_minus_bottom_ci,top_minus_random_pp,top_minus_random_p,top_minus_random_ci,top_minus_redundancy_matched_pp,top_minus_redundancy_matched_p,top_minus_redundancy_matched_ci
0,permutation,mcar,1.901745,0.729442,1.305542,0.833530,0.573297,0.544199,0.543241,0.584891,...,0.433867,1.172303,0.156250,"[-0.00, 2.59]",0.596202,0.296875,"[-0.16, 1.56]",1.068214,0.375000,"[-0.08, 2.52]"
1,permutation,mnar_strong,1.260096,0.461102,0.658630,0.596652,0.573297,0.544199,0.543241,0.584891,...,0.433867,0.798995,0.218750,"[-0.02, 2.11]",0.601467,0.296875,"[-0.12, 1.63]",0.663444,0.468750,"[-0.27, 2.01]"
2,xgb_shap,mcar,2.435712,0.603562,0.924089,0.497900,0.578941,0.492294,0.549191,0.597856,...,0.500462,1.832149,0.078125,"[0.50, 3.44]",1.511622,0.078125,"[0.35, 2.89]",1.937811,0.046875,"[0.48, 3.74]"
3,xgb_shap,mnar_strong,1.553674,0.325602,0.673616,0.460806,0.578941,0.492294,0.549191,0.597856,...,0.500462,1.228072,0.015625,"[0.34, 2.39]",0.880058,0.046875,"[0.17, 1.84]",1.092868,0.078125,"[0.18, 2.37]"


,imputer,target,converged_share,median_iter,warning_share,drop_pp,std_rmse,imputer_s
0,iterative,block3,0.285714,5.0,0.714286,2.212603,0.896454,0.038622
1,iterative,single,0.468254,5.0,0.531746,0.612734,0.933232,0.015968
2,iterative_allfeat,block3,1.000000,3.5,0.000000,2.209013,0.877523,0.051561
3,iterative_allfeat,single,1.000000,2.0,0.000000,0.620638,0.885954,0.010063
4,iterative_mi25,block3,0.428571,25.0,0.571429,2.187445,0.890340,0.171243
5,iterative_mi25,single,0.507937,13.5,0.492063,0.619484,0.924127,0.047534
6,mean,block3,NaN,NaN,0.000000,2.038026,1.180547,0.002059
7,mean,single,NaN,NaN,0.000000,0.498370,1.395588,0.002336
8,mice5,block3,NaN,10.0,0.000000,2.227105,0.933233,0.987482
9,mice5,single,NaN,10.0,0.000000,0.507157,0.930121,0.342933


,target,mechanism,contrast,n_datasets,mean_diff_pp,ci_low_pp,ci_high_pp,wilcoxon_p,datasets_positive,max_abs_dataset_diff_pp
0,block3,mar_redundant,iterative_mi25 - iterative,7,-0.198498,-0.434728,0.048116,0.312500,2,0.631004
1,block3,mar_redundant,rf_iterative_mi25 - rf_iterative,7,-0.168164,-0.321076,0.030542,0.218750,1,0.419161
2,block3,mar_redundant,iterative_allfeat - iterative,7,0.018700,-0.153152,0.185009,1.000000,3,0.388219
3,block3,mar_redundant,rf_iterative_allfeat - rf_iterative,7,-0.388384,-0.555516,-0.197548,0.031250,1,0.702529
4,block3,mar_redundant,missforest - rf_iterative,7,-0.180088,-0.534988,0.206171,0.375000,2,0.829335
5,block3,mar_redundant,mice5 - iterative,7,0.013375,-0.435015,0.446132,0.937500,3,0.980218
6,block3,mar_redundant,missforest - mean,7,-0.172896,-0.478862,0.178117,0.296875,2,0.657665
7,block3,mar_redundant,mice5 - mean,7,0.103966,-0.423067,0.595697,0.937500,4,1.170144
8,block3,mcar,iterative_mi25 - iterative,7,0.004051,-0.142242,0.121112,0.812500,3,0.375121
9,block3,mcar,rf_iterative_mi25 - rf_iterative,7,0.007219,-0.230737,0.246063,1.000000,3,0.626526


clf,imputer,hgb,lr,rf,xgb,xgb_tuned
0,iterative,0.354,0.497,0.364,0.361,0.244
1,knn,0.473,0.209,0.306,0.343,0.260
2,mean,0.222,0.918,0.051,0.138,-0.009
3,median_indicator,0.166,-0.253,-0.045,0.096,-0.065
4,native,0.089,NaN,NaN,0.109,-0.078
5,rf_iterative,0.326,0.223,0.243,0.274,0.144


... (8 outputs left out)

,dataset,fold,mechanism,imputer,mean,std,min,max
0,banknote,0,mcar,knn,99.328,0.000,99.328,99.328
1,banknote,0,mcar,missforest,99.238,0.217,98.879,99.552
2,banknote,0,mcar,rf_iterative,99.238,0.116,99.103,99.328
3,breast_cancer,0,mcar,knn,97.146,0.000,97.146,97.146
4,breast_cancer,0,mcar,missforest,97.146,0.000,97.146,97.146
5,breast_cancer,0,mcar,rf_iterative,97.088,0.184,96.565,97.146
6,digits,0,mcar,knn,97.006,0.000,97.006,97.006
7,digits,0,mcar,missforest,96.114,0.273,95.407,96.411
8,digits,0,mcar,rf_iterative,96.375,0.349,95.811,96.815
9,ionosphere,0,mcar,knn,94.231,0.000,94.231,94.231


,dataset,fold,mechanism,forest_prediction,repeats,distinct_macro_f1,min_macro_f1,max_macro_f1,max_imputed_diff
0,banknote,0,mcar,multi-threaded (original),8,1,0.993272,0.993272,3.552714e-15
1,banknote,0,mcar,serial (deterministic),8,1,0.993272,0.993272,0.000000e+00
2,breast_cancer,0,mcar,multi-threaded (original),8,1,0.965646,0.965646,1.387779e-17
3,breast_cancer,0,mcar,serial (deterministic),8,1,0.965646,0.965646,0.000000e+00
4,digits,0,mcar,multi-threaded (original),8,1,0.966136,0.966136,3.552714e-15
5,digits,0,mcar,serial (deterministic),8,1,0.966136,0.966136,0.000000e+00
6,ionosphere,0,mcar,multi-threaded (original),8,1,0.942308,0.942308,2.220446e-16
7,ionosphere,0,mcar,serial (deterministic),8,1,0.942308,0.942308,0.000000e+00
8,magic,0,mcar,multi-threaded (original),8,1,0.843122,0.843122,8.881784e-16
9,magic,0,mcar,serial (deterministic),8,1,0.843122,0.843122,0.000000e+00


,archived_masks,note
0,1098,original results not attached


Wrote /kaggle/working/supplementary/summary/response_numbers.md


## Manifest and download bundle

In [16]:
CFG_NAMES = ['FAST', 'N_THREADS', 'RUN', 'SEED', 'FOLDS', 'REPEATS', 'MASK_SEED', 'RATE', 'MAX_ROWS', 'FOLD_IDS', 'IMP_REPEATS',
             'IMP_PERM_REPEATS', 'MAX_MEMBERS', 'XGB_SEEDS', 'TUNE_CONFIGS', 'MICE_M', 'SAVE_MODEL_FOLDS', 'SAVE_MODEL_EXPS', 'DATASETS', 'XGB_DEFAULT']
manifest = {'versions': VERSIONS, 'hardware': hardware(), 'config': {k: globals()[k] for k in CFG_NAMES},
            'imputers': IMPUTER_INFO, 'dataset_fingerprints': {n: {c: v['fingerprint'] for c, v in DATA[n]['versions'].items()}
                                                               for n in DATASETS}, 'files': {}}
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.suffix != '.zip':
        manifest['files'][str(p.relative_to(OUT))] = hashlib.sha256(p.read_bytes()).hexdigest()
atomic_json(manifest, OUT / 'manifest.json')
zip_path = OUT.parent / f'{OUT.name}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): zf.write(p, p.relative_to(OUT.parent))
print('Bundle:', zip_path, f'{zip_path.stat().st_size / 1e6:.1f} MB')

Bundle: /kaggle/working/supplementary_results.zip 261.9 MB


In [18]:
from IPython.display import FileLink
import os

os.chdir('/kaggle/working')
FileLink('supplementary_results.zip')


/kaggle/working/supplementary_results.zip